# PREBORN: the Barts extension

This notebook builds the Barts (R1H) PREBORN dataset. It starts from the MSDS dataset that `10_preborn_msds`
published, keeps every MSDS row exactly as published, and adds what Barts' silver layer holds beyond MSDS:

- **fills**: a few pregnancy and infant values MSDS left empty (marital status, pre-pregnancy BMI, birth weight);
- **neonatal unit episodes** from the BadgerNet neonatal tables: stays, examinations and deaths;
- **maternity-relevant EHR facts** for each mother and baby within a time window: conditions, procedures,
  medicines given, pathology results, vital signs, allergies, devices, the encounters they belong to, and deaths.

It reads `4_prod.silver` only (plus the OMOP vocabulary). Silver already carries mapped OMOP concepts and their
mapping provenance, which are passed through.

**How it runs, top to bottom**

1. Settings, and a check that the MSDS dataset was published without hard QA failures.
2. Small helpers (the same as in `10_preborn_msds`).
3. The shape of every output table.
4. Seed: copy the published MSDS rows, pinned to one table version each.
5. Link PREBORN people to silver people, without using any identifier.
6. Neonatal unit episodes.
7. Fills for empty MSDS values.
8. The maternity concept set and the EHR facts.
9. Cleaning.
10. Assemble the OMOP tables; observation periods, episode links, CDM source and the bundled vocabulary.
11. Quality checks, including proof that every MSDS row is unchanged.
12. Publish (locked, all-or-nothing, rolled back if the checks fail afterwards) and tidy up.

**Working schemas** are named after the run tag, as in `10_preborn_msds`: `{target}_pub_{run_tag}` and
`{target}_pub_{run_tag}_stg`. Both are dropped at the end of a successful run.

## 1. Settings

In [0]:
import json
import re
import uuid
from datetime import date, datetime, timezone

dbutils.widgets.text("msds_schema", "5_projects.preborn_msds")   # where 10_preborn_msds published
dbutils.widgets.text("target", "5_projects.preborn")              # the published Barts PREBORN schema
dbutils.widgets.text("release_date", "")                     # blank = today; rows dated after it are dropped
dbutils.widgets.text("run_tag", "")                          # blank = a new random tag
dbutils.widgets.dropdown("keep_working_schemas", "no", ["no", "yes"])

SITE_ID = "R1H"
MSDS_SCHEMA = dbutils.widgets.get("msds_schema").strip()
TARGET = dbutils.widgets.get("target").strip()
RELEASE_DATE = dbutils.widgets.get("release_date").strip() or date.today().isoformat()
KEEP_WORKING_SCHEMAS = dbutils.widgets.get("keep_working_schemas") == "yes"
assert re.fullmatch(r"\d{4}-\d{2}-\d{2}", RELEASE_DATE), f"release_date must be YYYY-MM-DD, got {RELEASE_DATE!r}"

RUN_TAG = dbutils.widgets.get("run_tag").strip().lower() or f"r1hb_{uuid.uuid4().hex[:12]}"
assert re.fullmatch(r"[a-z0-9_]+", RUN_TAG), f"run_tag may only use a-z, 0-9 and _: {RUN_TAG!r}"

PUB = f"{TARGET}_pub_{RUN_TAG}"     # finished OMOP tables for this run, before publishing
STG = f"{PUB}_stg"                  # working tables for this run
VOCAB = "4_prod.omop"
SILVER = "4_prod.silver"

print(f"{MSDS_SCHEMA} -> {TARGET} | release {RELEASE_DATE} | run tag {RUN_TAG}")

In [0]:
# The extension only builds on a validated MSDS publish: the site must be there, and the latest QA run must have
# no hard failures. (Section 4 checks this again, against the exact run it seeds from.)
msds_people = spark.sql(f"SELECT count(*) FROM {MSDS_SCHEMA}.person WHERE site_id = '{SITE_ID}'").first()[0]
assert msds_people > 0, f"{MSDS_SCHEMA} has no {SITE_ID} people: run 10_preborn_msds first"
last_qa = spark.sql(f"""SELECT run_tag, count_if(NOT ok AND severity = 'hard') AS hard
  FROM {MSDS_SCHEMA}._qa_results WHERE site_id = '{SITE_ID}' GROUP BY run_tag
  ORDER BY max(checked_at) DESC LIMIT 1""").first()
assert last_qa is not None and last_qa.hard == 0, f"latest MSDS QA run {last_qa and last_qa.run_tag} has hard failures"
print(f"MSDS {SITE_ID}: {msds_people} people; latest QA run {last_qa.run_tag} clean")

## 2. Small helpers

*The same code as section 2 of `10_preborn_msds`; `tests/test_preborn_notebooks.py` checks the two copies
match.*

**Ids.** Every PREBORN id (person, pregnancy, visit, fact, ...) is *calculated* from the source key rather
than counted. The same MSDS row therefore gets the same id on every run and at every site's release, which
downstream products (the synthetic data, linked extracts) rely on. Ids must never change, so the recipe in
`mint_id` must never change either.

In [0]:
def sql_str(value):
    """Escape a Python value for use inside a SQL '...' string literal (doubles any single quote)."""
    return str(value).replace("'", "''")


SITE = sql_str(SITE_ID)   # the site id, ready to drop into SQL as '{SITE}'


def _normalised_key_part(sql_expr):
    """SQL for one id key part: trimmed text, with blank or NULL replaced by '~' so it still hashes."""
    return f"coalesce(nullif(trim(cast({sql_expr} as string)), ''), '~')"


def mint_id(kind, *key_sql):
    """SQL expression for a deterministic PREBORN id.

    The id is SHA-256 of 'kind|site|key1|key2|...', with the first 15 hex digits read as a number.
    `kind` names what is being identified ('person', 'pregnancy', 'visit', ...), so different kinds of thing
    never share an id. `key_sql` are SQL expressions (column names, or quoted literals such as "'X1'"),
    not Python values.
    """
    site_literal = "'" + sql_str(SITE_ID) + "'"
    parts = [f"'{kind}'", _normalised_key_part(site_literal)]
    for key in key_sql:
        parts.append(_normalised_key_part(key))
    joined = ", ".join(parts)
    return f"conv(substr(sha2(concat_ws('|', {joined}), 256), 1, 15), 16, 10)"


def check_mint_id_matches_python():
    """Stop the run if Spark's id recipe ever stops matching the same recipe done in plain Python."""
    import hashlib
    key = f"pregnancy|{SITE_ID}|TESTKEY"
    expected = int(hashlib.sha256(key.encode()).hexdigest()[:15], 16)
    got = spark.sql(f"SELECT {mint_id('pregnancy', chr(39) + 'TESTKEY' + chr(39))} AS v").first()["v"]
    assert int(got) == expected, f"id minting has changed: Spark gave {got}, Python gave {expected}"
    print("id minting check passed for site", SITE_ID)


def scalar(sql):
    """Run a query that returns one value and return that value."""
    return spark.sql(sql).first()[0]


def show(sql):
    """Run a query and print the result as a small text table."""
    print(spark.sql(sql).toPandas().to_string(index=False))


def schema_exists(schema):
    """True when catalog.schema exists."""
    catalog, name = schema.split(".", 1)
    return spark.sql(f"SHOW SCHEMAS IN `{catalog}` LIKE '{sql_str(name)}'").count() > 0


def table_columns(schema, table):
    """The column names of schema.table, in table order."""
    catalog, name = schema.split(".", 1)
    rows = spark.sql(f"""
      SELECT column_name FROM {catalog}.information_schema.columns
      WHERE table_schema = '{name}' AND table_name = '{table}'
      ORDER BY ordinal_position""").collect()
    columns = [row["column_name"] for row in rows]
    if not columns:
        raise RuntimeError(f"no columns found for {schema}.{table}")
    return columns


def latest_version(full_table_name):
    """The table's current Delta version."""
    return int(spark.sql(f"DESCRIBE HISTORY {full_table_name} LIMIT 1").first()["version"])


def assert_unique(table, id_column, where="true"):
    """Stop the run if id_column is not unique in table (an id collision would corrupt every join)."""
    rows, distinct_ids = spark.sql(
        f"SELECT count(*), count(DISTINCT {id_column}) FROM {table} WHERE {where}").first()
    assert rows == distinct_ids, f"{table}.{id_column} is not unique: {rows} rows, {distinct_ids} ids"
    return rows


check_mint_id_matches_python()

## 3. The shape of every output table

The OMOP CDM v5.4 tables plus the PREBORN `pregnancy` and `infant` tables, exactly as in section 3 of
`10_preborn_msds` (copied; the test suite checks the copies match), plus the tables only this extension writes:
`drug_exposure`, `device_exposure`, the maternity concept set and the EHR mapping provenance.

In [0]:
CDM_TABLES = {
    "person": """
      person_id BIGINT, gender_concept_id BIGINT, year_of_birth BIGINT,
      month_of_birth BIGINT, day_of_birth BIGINT, birth_datetime TIMESTAMP,
      race_concept_id BIGINT, ethnicity_concept_id BIGINT, location_id BIGINT,
      provider_id BIGINT, care_site_id BIGINT, person_source_value STRING,
      gender_source_value STRING, gender_source_concept_id BIGINT,
      race_source_value STRING, race_source_concept_id BIGINT,
      ethnicity_source_value STRING, ethnicity_source_concept_id BIGINT,
      site_id STRING, person_role STRING""",
    "observation_period": """
      observation_period_id BIGINT, person_id BIGINT,
      observation_period_start_date DATE, observation_period_end_date DATE,
      period_type_concept_id BIGINT, site_id STRING""",
    "condition_occurrence": """
      condition_occurrence_id BIGINT, person_id BIGINT, condition_concept_id BIGINT,
      condition_start_date DATE, condition_start_datetime TIMESTAMP,
      condition_end_date DATE, condition_end_datetime TIMESTAMP,
      condition_type_concept_id BIGINT, condition_status_concept_id BIGINT,
      stop_reason STRING, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, condition_source_value STRING,
      condition_source_concept_id BIGINT, condition_status_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING""",
    "procedure_occurrence": """
      procedure_occurrence_id BIGINT, person_id BIGINT, procedure_concept_id BIGINT,
      procedure_date DATE, procedure_datetime TIMESTAMP,
      procedure_end_date DATE, procedure_end_datetime TIMESTAMP,
      procedure_type_concept_id BIGINT, modifier_concept_id BIGINT, quantity BIGINT,
      provider_id BIGINT, visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      procedure_source_value STRING, procedure_source_concept_id BIGINT,
      modifier_source_value STRING,
      site_id STRING, pregnancy_id BIGINT, source_table STRING""",
    "measurement": """
      measurement_id BIGINT, person_id BIGINT, measurement_concept_id BIGINT,
      measurement_date DATE, measurement_datetime TIMESTAMP, measurement_time STRING,
      measurement_type_concept_id BIGINT, operator_concept_id BIGINT,
      value_as_number DOUBLE, value_as_concept_id BIGINT, unit_concept_id BIGINT,
      range_low DOUBLE, range_high DOUBLE, provider_id BIGINT,
      visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      measurement_source_value STRING, measurement_source_concept_id BIGINT,
      unit_source_value STRING, unit_source_concept_id BIGINT, value_source_value STRING,
      measurement_event_id BIGINT, meas_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING""",
    "observation": """
      observation_id BIGINT, person_id BIGINT, observation_concept_id BIGINT,
      observation_date DATE, observation_datetime TIMESTAMP,
      observation_type_concept_id BIGINT, value_as_number DOUBLE,
      value_as_string STRING, value_as_concept_id BIGINT, qualifier_concept_id BIGINT,
      unit_concept_id BIGINT, provider_id BIGINT, visit_occurrence_id BIGINT,
      visit_detail_id BIGINT, observation_source_value STRING,
      observation_source_concept_id BIGINT, unit_source_value STRING,
      qualifier_source_value STRING, value_source_value STRING,
      observation_event_id BIGINT, obs_event_field_concept_id BIGINT,
      site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING, source_table STRING""",
    "visit_occurrence": """
      visit_occurrence_id BIGINT, person_id BIGINT, visit_concept_id BIGINT, visit_start_date DATE,
      visit_start_datetime TIMESTAMP, visit_end_date DATE, visit_end_datetime TIMESTAMP,
      visit_type_concept_id BIGINT, provider_id BIGINT, care_site_id BIGINT, visit_source_value STRING,
      visit_source_concept_id BIGINT, admitted_from_concept_id BIGINT, admitted_from_source_value STRING,
      discharged_to_concept_id BIGINT, discharged_to_source_value STRING,
      preceding_visit_occurrence_id BIGINT, site_id STRING, pregnancy_id BIGINT""",
    "visit_detail": """
      visit_detail_id BIGINT, person_id BIGINT, visit_detail_concept_id BIGINT,
      visit_detail_start_date DATE, visit_detail_start_datetime TIMESTAMP, visit_detail_end_date DATE,
      visit_detail_end_datetime TIMESTAMP, visit_detail_type_concept_id BIGINT, provider_id BIGINT,
      care_site_id BIGINT, visit_detail_source_value STRING, visit_detail_source_concept_id BIGINT,
      admitted_from_concept_id BIGINT, admitted_from_source_value STRING, discharged_to_source_value STRING,
      discharged_to_concept_id BIGINT, preceding_visit_detail_id BIGINT, parent_visit_detail_id BIGINT,
      visit_occurrence_id BIGINT, site_id STRING, pregnancy_id BIGINT, enrichment_tag STRING,
      source_table STRING""",
    "death": """
      person_id BIGINT, death_date DATE, death_datetime TIMESTAMP, death_type_concept_id BIGINT,
      cause_concept_id BIGINT, cause_source_value STRING, cause_source_concept_id BIGINT, site_id STRING""",
    "location": """
      location_id BIGINT, address_1 STRING, address_2 STRING, city STRING, state STRING, zip STRING,
      county STRING, location_source_value STRING, country_concept_id BIGINT, country_source_value STRING,
      latitude DOUBLE, longitude DOUBLE, site_id STRING""",
    "episode": """
      episode_id BIGINT, person_id BIGINT, episode_concept_id BIGINT,
      episode_start_date DATE, episode_start_datetime TIMESTAMP,
      episode_end_date DATE, episode_end_datetime TIMESTAMP, episode_parent_id BIGINT,
      episode_number BIGINT, episode_object_concept_id BIGINT,
      episode_type_concept_id BIGINT, episode_source_value STRING,
      episode_source_concept_id BIGINT, pregnancy_id BIGINT, site_id STRING""",
    "episode_event": """
      episode_id BIGINT, event_id BIGINT, episode_event_field_concept_id BIGINT,
      source_table STRING, site_id STRING""",
    "fact_relationship": """
      domain_concept_id_1 BIGINT, fact_id_1 BIGINT, domain_concept_id_2 BIGINT,
      fact_id_2 BIGINT, relationship_concept_id BIGINT, site_id STRING""",
    "cdm_source": """
      cdm_source_name STRING, cdm_source_abbreviation STRING, cdm_holder STRING,
      source_description STRING, source_documentation_reference STRING,
      cdm_etl_reference STRING, source_release_date DATE, cdm_release_date DATE,
      cdm_version STRING, cdm_version_concept_id BIGINT, vocabulary_version STRING""",
    # PREBORN maternity extension: one row per pregnancy, one row per infant.
    "pregnancy": """
      person_id BIGINT, pregnancy_id BIGINT,
      pregnancy_start_date DATE, pregnancy_end_date DATE,
      gestational_length_in_day INT,
      pregnancy_outcome_concept_id BIGINT, pregnancy_mode_delivery_concept_id BIGINT,
      pregnancy_single_concept_id BIGINT, pregnancy_marital_status_concept_id BIGINT,
      pregnancy_number_fetuses INT, pregnancy_number_liveborn INT,
      prev_pregnancy_parity_concept_id BIGINT, prev_pregnancy_gravidity INT,
      prev_livebirth_number INT, prev_still_misc_number INT,
      pre_pregnancy_bmi DOUBLE, pregnancy_folic_concept_id BIGINT,
      pregnancy_outcome_source_value STRING, pregnancy_mode_delivery_source_value STRING,
      site_id STRING,
      pre_pregnancy_bmi_source STRING, pregnancy_marital_status_source_value STRING""",
    "infant": """
      pregnancy_id BIGINT, infant_id BIGINT,
      birth_outcome_concept_id BIGINT, birth_weight DOUBLE,
      birth_con_malformation_concept_id BIGINT, birth_apgar INT,
      labour_delivery_id STRING, site_id STRING,
      birth_weight_source STRING, birth_apgar_source STRING""",
    # The bundled vocabulary (filled in section 11).
    "concept": """
      concept_id BIGINT, concept_name STRING, domain_id STRING, vocabulary_id STRING,
      concept_class_id STRING, standard_concept STRING, concept_code STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING""",
    "concept_relationship": """
      concept_id_1 BIGINT, concept_id_2 BIGINT, relationship_id STRING,
      valid_start_date DATE, valid_end_date DATE, invalid_reason STRING""",
    "concept_ancestor": """
      ancestor_concept_id BIGINT, descendant_concept_id BIGINT,
      min_levels_of_separation BIGINT, max_levels_of_separation BIGINT""",
    "concept_synonym": """
      concept_id BIGINT, concept_synonym_name STRING, language_concept_id BIGINT""",
    "vocabulary": """
      vocabulary_id STRING, vocabulary_name STRING, vocabulary_reference STRING,
      vocabulary_version STRING, vocabulary_concept_id BIGINT""",
    "domain": """
      domain_id STRING, domain_name STRING, domain_concept_id BIGINT""",
    "concept_class": """
      concept_class_id STRING, concept_class_name STRING, concept_class_concept_id BIGINT""",
    "relationship": """
      relationship_id STRING, relationship_name STRING, is_hierarchical STRING,
      defines_ancestry STRING, reverse_relationship_id STRING, relationship_concept_id BIGINT""",
    # How every MSDS spec question, answer and unit was mapped to a concept, with its method (section 5).
    "_msds_concept_map": """
      site_id STRING, rule_id STRING, role STRING, src_code STRING, src_text STRING,
      concept_id BIGINT, domain_id STRING, method STRING, model_version STRING,
      cosine DOUBLE, margin DOUBLE, rule_ref STRING""",
}


def column_names(table):
    """The column names of a CDM_TABLES entry, in order."""
    return [column.split()[0] for column in CDM_TABLES[table].split(",")]


def create_cdm_tables(schema):
    """Create schema and every CDM_TABLES table in it (tables that already exist are left alone)."""
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
    for table, columns in CDM_TABLES.items():
        spark.sql(f"CREATE TABLE IF NOT EXISTS {schema}.{table} ({columns}) USING DELTA")


# The tables this notebook publishes, per site, to the target.
MSDS_PUBLISH_TABLES = [
    "person", "observation_period",
    "condition_occurrence", "procedure_occurrence", "measurement", "observation",
    "visit_occurrence", "visit_detail", "death", "location",
    "episode", "episode_event", "fact_relationship",
    "pregnancy", "infant", "_msds_concept_map",
]

# The bundled vocabulary tables (rebuilt for the whole target, not per site).
VOCAB_TABLES = [
    "concept", "concept_relationship", "concept_ancestor", "concept_synonym",
    "vocabulary", "domain", "concept_class", "relationship",
]


# Tables only the Barts extension writes.
EXTENSION_TABLES = {
    "drug_exposure": """
      drug_exposure_id BIGINT, person_id BIGINT, drug_concept_id BIGINT,
      drug_exposure_start_date DATE, drug_exposure_start_datetime TIMESTAMP,
      drug_exposure_end_date DATE, drug_exposure_end_datetime TIMESTAMP, verbatim_end_date DATE,
      drug_type_concept_id BIGINT, stop_reason STRING, refills BIGINT, quantity DOUBLE,
      days_supply BIGINT, sig STRING, route_concept_id BIGINT, lot_number STRING,
      provider_id BIGINT, visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      drug_source_value STRING, drug_source_concept_id BIGINT, route_source_value STRING,
      dose_unit_source_value STRING, site_id STRING, pregnancy_id BIGINT, source_table STRING""",
    "device_exposure": """
      device_exposure_id BIGINT, person_id BIGINT, device_concept_id BIGINT,
      device_exposure_start_date DATE, device_exposure_start_datetime TIMESTAMP,
      device_exposure_end_date DATE, device_exposure_end_datetime TIMESTAMP,
      device_type_concept_id BIGINT, unique_device_id STRING, production_id STRING,
      quantity BIGINT, provider_id BIGINT, visit_occurrence_id BIGINT, visit_detail_id BIGINT,
      device_source_value STRING, device_source_concept_id BIGINT, unit_concept_id BIGINT,
      unit_source_value STRING, unit_source_concept_id BIGINT, site_id STRING, pregnancy_id BIGINT,
      source_table STRING""",
    # Each concept the EHR lane admits, and the anchors and groups that admitted it (section 8).
    "_concept_set": """
      site_id STRING, concept_id BIGINT, role STRING, anchors ARRAY<STRING>, groups ARRAY<STRING>""",
    # The mapping provenance of every row this extension adds (method, version, cosine, rule).
    "_ehr_provenance": """
      site_id STRING, cdm_table STRING, cdm_id BIGINT, map_method STRING,
      map_version STRING, map_cosine DOUBLE, map_rule_id STRING""",
}
CDM_TABLES.update(EXTENSION_TABLES)

# Everything 10_preborn_msds publishes, plus the extension's tables.
BEYOND_PUBLISH_TABLES = MSDS_PUBLISH_TABLES + ["drug_exposure", "device_exposure", "_concept_set", "_ehr_provenance"]


def record_dropped(select_sql):
    """Append rows to STG._dropped. select_sql must return (rule_id, fact_natural_key, reason)."""
    spark.sql(f"INSERT INTO {STG}._dropped (rule_id, fact_natural_key, reason) {select_sql}")

In [0]:
# Create this run's scratch schemas. A run tag is used once: an existing schema means a clash.
assert not schema_exists(PUB) and not schema_exists(STG), f"working schemas for run tag {RUN_TAG} already exist"
create_cdm_tables(PUB)
create_cdm_tables(STG)
spark.sql(f"CREATE TABLE {STG}._dropped (rule_id STRING, fact_natural_key STRING, reason STRING) USING DELTA")
print("created", PUB, "and", STG)

## 4. Seed from the published MSDS dataset

The extension starts from exactly what `10_preborn_msds` published for R1H: the latest published run (refused
if that run had any hard QA failure), each table read at the Delta version current at seed time. The versions
are kept in `STG._seed_state`, so the preservation check (section 11) compares against the same snapshot.

**MSDS rows are never changed.** The only exception is the fills in `FILL_COLUMNS`, which may put a silver
value where MSDS left NULL (section 7). Every other published MSDS value is protected, and section 11 proves it.

In [0]:
# Columns the extension may fill, and only where MSDS left them NULL.
FILL_COLUMNS = {
    "pregnancy": ["pre_pregnancy_bmi", "pre_pregnancy_bmi_source", "pregnancy_marital_status_concept_id",
                  "pregnancy_marital_status_source_value", "prev_pregnancy_gravidity",
                  "prev_pregnancy_parity_concept_id"],
    "infant": ["birth_weight", "birth_weight_source", "birth_apgar", "birth_apgar_source"],
}

# The row key of each seeded table, for the preservation check. None = no key: the check then compares whole
# rows as a multiset. observation_period and episode_event are rebuilt by this notebook and checked structurally.
SEED_KEYS = {
    "person": "person_id", "condition_occurrence": "condition_occurrence_id",
    "procedure_occurrence": "procedure_occurrence_id", "measurement": "measurement_id",
    "observation": "observation_id", "visit_occurrence": "visit_occurrence_id",
    "visit_detail": "visit_detail_id", "death": "person_id", "location": "location_id",
    "episode": "episode_id", "pregnancy": "pregnancy_id", "infant": "infant_id",
    "fact_relationship": None, "_msds_concept_map": None,
}
assert set(SEED_KEYS) | {"observation_period", "episode_event"} == set(MSDS_PUBLISH_TABLES)


def seed_from_msds():
    """Copy this site's rows of every MSDS publish table from MSDS_SCHEMA into STG, pinned to one version each."""
    runs = spark.sql(f"""SELECT run_tag, max(source_release) AS release, max(checked_at) AS at,
        count_if(NOT ok AND severity = 'hard') AS hard
      FROM {MSDS_SCHEMA}._qa_results WHERE site_id = '{SITE}' GROUP BY run_tag
      HAVING count_if(scope = 'published') > 0 ORDER BY at DESC LIMIT 1""").collect()
    assert runs, f"no published MSDS run for {SITE_ID} in {MSDS_SCHEMA}._qa_results"
    run = runs[0]
    assert run.hard == 0, f"latest published MSDS run {run.run_tag} has {run.hard} hard QA failures; refusing to seed"

    versions = {table: latest_version(f"{MSDS_SCHEMA}.{table}") for table in MSDS_PUBLISH_TABLES}
    for table in MSDS_PUBLISH_TABLES:
        columns = ", ".join(f"`{c}`" for c in table_columns(MSDS_SCHEMA, table))
        spark.sql(f"""INSERT INTO {STG}.{table} ({columns})
          SELECT {columns} FROM {MSDS_SCHEMA}.{table} VERSION AS OF {versions[table]} WHERE site_id = '{SITE}'""")
    spark.sql(f"""CREATE OR REPLACE TABLE {STG}._seed_state AS SELECT
      '{MSDS_SCHEMA}' AS msds_schema, '{SITE}' AS site_id, '{run.run_tag}' AS msds_run_tag,
      DATE'{run.release}' AS source_release, '{json.dumps(versions, sort_keys=True)}' AS versions,
      current_timestamp() AS seeded_at""")
    print(f"seeded {STG} from {MSDS_SCHEMA} run {run.run_tag} (release {run.release}): {versions}")


seed_from_msds()

## 5. Link PREBORN people to silver people

The published MSDS dataset holds no NHS number or local patient id, and this notebook reads silver only, so the
link uses **keys both sides already share**, never an identifier:

- **Mothers.** Silver's maternity care contacts carry the MSDS `PREGNANCYID`. Minting PREBORN's pregnancy id from
  it gives the PREBORN pregnancy, and so its mother. Used only where each silver pregnancy has exactly one silver
  person.
- **Babies.** From silver's baby delivery records on the same pregnancy: when the pregnancy has one PREBORN baby
  and one silver baby (`singleton`), or when a baby's birth time is the nearest on both sides within 5 minutes
  with no tie (`birth_time`; twins are born minutes apart).

Every PREBORN person gets one row in `STG._bridge` (never published). `silver_person_id` is NULL unless the link
is unambiguous, and `method` says why:

| method | meaning |
|---|---|
| pregnancy_id, singleton, birth_time | linked |
| lane_conflict | a mother's pregnancies point at different silver people |
| ambiguous | the pregnancy has silver babies but no unique match |
| unmatched | nothing to match |
| shared_silver | one silver person would serve two PREBORN people, so neither is linked |

Section 11 requires at least 90% of mothers and 65% of babies linked: a much lower rate means a broken key.

In [0]:
BRIDGE_BIRTH_TOLERANCE_SEC = 300
BRIDGE_FLOORS = {"mother": 0.90, "infant": 0.65}


def build_person_bridge():
    """Create STG._bridge: one row per PREBORN person, with its silver person where the link is unambiguous."""
    pregnancy_id = f"CAST({mint_id('pregnancy', 'x.pregnancy_id')} AS BIGINT)"
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._bridge AS
    WITH
    -- silver pregnancy -> silver person, only where the pregnancy has exactly one person
    cc AS (
      SELECT pregnancy_id, min(person_id) AS spid FROM {SILVER}.clinical_maternity_care_contact
      WHERE record_status = 'active' AND pregnancy_id IS NOT NULL AND person_id IS NOT NULL
      GROUP BY pregnancy_id HAVING count(DISTINCT person_id) = 1),
    cck AS (SELECT {pregnancy_id} AS pid, x.spid FROM cc x),
    mom_preg AS (
      SELECT p.person_id, k.spid FROM {MSDS_SCHEMA}.pregnancy p
      LEFT JOIN cck k ON k.pid = p.pregnancy_id
      WHERE p.site_id = '{SITE}' AND p.person_id IS NOT NULL),
    mom AS (
      SELECT person_id, 'mother' AS role,
             CASE WHEN count(DISTINCT spid) = 1 THEN min(spid) END AS spid,
             CASE WHEN count(DISTINCT spid) = 1 THEN 'pregnancy_id'
                  WHEN count(DISTINCT spid) > 1 THEN 'lane_conflict' ELSE 'unmatched' END AS method
      FROM mom_preg GROUP BY person_id),
    -- silver babies per silver pregnancy (one row per baby)
    sb AS (
      SELECT pregnancy_id, baby_person_id, min(event_datetime) AS ev FROM {SILVER}.clinical_baby_delivery
      WHERE record_status = 'active' AND pregnancy_id IS NOT NULL AND baby_person_id IS NOT NULL
      GROUP BY pregnancy_id, baby_person_id),
    sbk AS (SELECT {pregnancy_id} AS pid, x.baby_person_id AS spid, x.ev,
                   count(*) OVER (PARTITION BY x.pregnancy_id) AS n_sb FROM sb x),
    inf AS (
      SELECT i.infant_id AS person_id, i.pregnancy_id, pe.birth_datetime AS bd,
             count(*) OVER (PARTITION BY i.pregnancy_id) AS n_inf
      FROM {MSDS_SCHEMA}.infant i
      LEFT JOIN {MSDS_SCHEMA}.person pe ON pe.person_id = i.infant_id AND pe.site_id = '{SITE}'
      WHERE i.site_id = '{SITE}'),
    -- birth-time candidates within the tolerance
    pair AS (
      SELECT i.person_id, b.spid, abs(unix_timestamp(b.ev) - unix_timestamp(i.bd)) AS d
      FROM inf i JOIN sbk b ON b.pid = i.pregnancy_id
      WHERE abs(unix_timestamp(b.ev) - unix_timestamp(i.bd)) <= {BRIDGE_BIRTH_TOLERANCE_SEC}),
    -- nearest on both sides, with no tie on either side
    pair_u AS (
      SELECT person_id, spid FROM (
        SELECT p.*,
               min(d) OVER (PARTITION BY person_id) AS min_i,
               count_if(true) OVER (PARTITION BY person_id, d) AS tie_i,
               min(d) OVER (PARTITION BY spid) AS min_s,
               count_if(true) OVER (PARTITION BY spid, d) AS tie_s FROM pair p)
      WHERE d = min_i AND d = min_s AND tie_i = 1 AND tie_s = 1),
    single AS (
      SELECT i.person_id, b.spid FROM inf i JOIN sbk b ON b.pid = i.pregnancy_id
      WHERE i.n_inf = 1 AND b.n_sb = 1),
    baby AS (
      SELECT i.person_id, 'infant' AS role, coalesce(sg.spid, pu.spid) AS spid,
             CASE WHEN sg.spid IS NOT NULL THEN 'singleton' WHEN pu.spid IS NOT NULL THEN 'birth_time'
                  WHEN EXISTS (SELECT 1 FROM sbk b WHERE b.pid = i.pregnancy_id) THEN 'ambiguous'
                  ELSE 'unmatched' END AS method
      FROM inf i
      LEFT JOIN single sg ON sg.person_id = i.person_id
      LEFT JOIN pair_u pu ON pu.person_id = i.person_id),
    allp AS (SELECT * FROM mom UNION ALL SELECT * FROM baby),
    shared AS (SELECT spid FROM allp WHERE spid IS NOT NULL GROUP BY spid HAVING count(*) > 1)
    SELECT a.person_id, a.role,
           CASE WHEN sh.spid IS NULL THEN a.spid END AS silver_person_id,
           CASE WHEN sh.spid IS NULL THEN a.method ELSE 'shared_silver' END AS method
    FROM allp a LEFT JOIN shared sh ON sh.spid = a.spid""")
    people = assert_unique(f"{STG}._bridge", "person_id")
    shared = scalar(f"""SELECT count(*) FROM (SELECT silver_person_id FROM {STG}._bridge
      WHERE silver_person_id IS NOT NULL GROUP BY 1 HAVING count(*) > 1)""")
    assert shared == 0, f"_bridge: {shared} silver people serve more than one PREBORN person"
    show(f"SELECT role, method, count(*) n FROM {STG}._bridge GROUP BY 1, 2 ORDER BY 1, 2")
    print("people in the bridge:", people)


build_person_bridge()

## 6. Neonatal unit episodes

From silver's BadgerNet neonatal tables (`clinical_neonatal_episode`, `clinical_neonatal_examination`).

**Which baby.** An episode is linked to a PREBORN infant by either route:

- `person_id`: the episode's silver person is a linked PREBORN baby (section 5);
- `mother_birth_time`: the episode's mother is a linked PREBORN mother, and one of her PREBORN babies was born
  within 5 minutes of the episode's birth time, nearest with no tie.

When both routes give an answer and disagree, the episode is left out (and counted).

**What is written**

- one `visit_detail` per episode (inpatient 9201; silver gives the unit level, not the MSDS critical-care flag,
  so the level goes in the source value), under the infant's visit that covers the admission date, or else
  under a new visit of its own (`SILVER-NNU`);
- head circumference from the newborn examination as a `measurement` (cm);
- each examined NIPE item as a `procedure` (heart, spine and hips have standard examination concepts; eyes and
  genitalia do not, so 0), with the finding in the source value;
- a `death` for "Died" outcomes where PREBORN has no death yet.

MSDS's own neonatal stays (msd402) stay as they are; where both describe the same stay, both are kept and the
overlap is reported. Every row gets a provenance row in `_ehr_provenance`.

In [0]:
NNU_DISCHARGED_TO = {"Home": 581476, "Ward": 8717}
# examination column -> (item label, standard examination concept)
NNU_EXAM_ITEMS = {"heart_finding": ("heart", 4107649), "spine_finding": ("spine", 4111530),
                  "hips_finding": ("hips", 4108627), "right_hip_finding": ("right hip", 4108627),
                  "eyes_finding": ("eyes", 0), "genitalia_finding": ("genitalia", 0)}
HEAD_CIRCUMFERENCE_RANGE = (20, 60)    # cm
NNU_DEATH_SOURCE = "NNU final outcome: Died"   # cause_source_value of a death from the neonatal unit

# Silver-derived rows are written straight into the staged tables, so every lane's checks and report rows are
# collected here and logged with the QA results (section 11).
BUILD_REPORT = []


def silver_id(kind, *key_sql):
    """A minted id as BIGINT, for rows built from silver."""
    return f"CAST({mint_id(kind, *key_sql)} AS BIGINT)"


def link_neonatal_episodes():
    """Create STG._nnu_link (episode -> PREBORN infant, pregnancy, method). Returns how many episodes on a linked
    baby were left out because the two routes disagreed."""
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._nnu_link AS
    WITH ep AS (SELECT entity_id, person_id, mother_person_id, birth_datetime
                FROM {SILVER}.clinical_neonatal_episode WHERE record_status = 'active'),
    inf AS (SELECT i.infant_id, i.pregnancy_id, p.person_id AS mom, pe.birth_datetime AS bd
            FROM {STG}.infant i JOIN {STG}.pregnancy p ON p.pregnancy_id = i.pregnancy_id
            JOIN {STG}.person pe ON pe.person_id = i.infant_id WHERE i.site_id = '{SITE}'),
    -- route 1: the episode's own person is a linked baby
    d AS (SELECT ep.entity_id, b.person_id AS infant_id FROM ep
          JOIN {STG}._bridge b ON b.role = 'infant' AND b.silver_person_id = ep.person_id),
    -- route 2: the mother's baby born nearest the episode's birth time
    c AS (SELECT ep.entity_id, inf.infant_id,
                 abs(unix_timestamp(inf.bd) - unix_timestamp(ep.birth_datetime)) AS dd
          FROM ep JOIN {STG}._bridge b ON b.role = 'mother' AND b.silver_person_id = ep.mother_person_id
          JOIN inf ON inf.mom = b.person_id
          WHERE abs(unix_timestamp(inf.bd) - unix_timestamp(ep.birth_datetime)) <= {BRIDGE_BIRTH_TOLERANCE_SEC}),
    m AS (SELECT entity_id, infant_id FROM (
            SELECT c.*, min(dd) OVER (PARTITION BY entity_id) AS md,
                   count(*) OVER (PARTITION BY entity_id, dd) AS td FROM c)
          WHERE dd = md AND td = 1),
    u AS (SELECT coalesce(d.entity_id, m.entity_id) AS entity_id, d.infant_id AS d_inf, m.infant_id AS m_inf
          FROM d FULL OUTER JOIN m ON m.entity_id = d.entity_id)
    SELECT u.entity_id, coalesce(u.d_inf, u.m_inf) AS infant_id, i.pregnancy_id,
           CASE WHEN u.d_inf IS NOT NULL THEN 'person_id' ELSE 'mother_birth_time' END AS method
    FROM u JOIN inf i ON i.infant_id = coalesce(u.d_inf, u.m_inf)
    WHERE u.d_inf IS NULL OR u.m_inf IS NULL OR u.d_inf = u.m_inf""")
    assert_unique(f"{STG}._nnu_link", "entity_id")
    return scalar(f"""SELECT count(*) FROM (
      SELECT ep.entity_id FROM {SILVER}.clinical_neonatal_episode ep
      JOIN {STG}._bridge b ON b.role = 'infant' AND b.silver_person_id = ep.person_id
      WHERE ep.record_status = 'active') x LEFT ANTI JOIN {STG}._nnu_link l ON l.entity_id = x.entity_id""")


def stage_neonatal_episodes():
    """Create STG._nnu_ep: each linked episode with its dates and outcome, and the infant's existing visit that
    covers the admission date (the latest-starting one; ties go to the lowest id), if any."""
    discharged_to = ("CASE e.final_nnu_outcome "
                     + " ".join(f"WHEN '{k}' THEN {v}" for k, v in NNU_DISCHARGED_TO.items()) + " ELSE 0 END")
    spark.sql(f"""
    CREATE OR REPLACE TABLE {STG}._nnu_ep AS
    WITH e AS (
      SELECT l.entity_id, l.infant_id, l.pregnancy_id, e.admit_datetime AS st,
             CASE WHEN e.discharge_datetime IS NULL OR e.discharge_datetime < e.admit_datetime
                  THEN e.admit_datetime ELSE e.discharge_datetime END AS en,
             e.final_nnu_outcome, {discharged_to} AS dto, e.unit_level
      FROM {STG}._nnu_link l JOIN {SILVER}.clinical_neonatal_episode e ON e.entity_id = l.entity_id
      WHERE e.record_status = 'active' AND e.admit_datetime IS NOT NULL),
    h AS (
      SELECT e.entity_id, v.visit_occurrence_id, row_number() OVER (PARTITION BY e.entity_id
               ORDER BY v.visit_start_date DESC NULLS LAST, v.visit_occurrence_id ASC) AS rn
      FROM e JOIN {STG}.visit_occurrence v ON v.person_id = e.infant_id AND v.site_id = '{SITE}'
       AND to_date(e.st) BETWEEN v.visit_start_date AND coalesce(v.visit_end_date, v.visit_start_date)
      WHERE v.visit_source_value IS DISTINCT FROM 'SILVER-NNU')
    SELECT e.*, h.visit_occurrence_id AS host_vid FROM e LEFT JOIN h ON h.entity_id = e.entity_id AND h.rn = 1""")


def add_neonatal_visits_and_stays():
    """A SILVER-NNU visit for each episode with no covering visit, and one visit_detail per episode."""
    own_visit_id = silver_id("silver-nnu-visit", "entity_id")
    spark.sql(f"""INSERT INTO {STG}.visit_occurrence (visit_occurrence_id, person_id, visit_concept_id, visit_start_date,
        visit_start_datetime, visit_end_date, visit_end_datetime, visit_type_concept_id, visit_source_value,
        visit_source_concept_id, discharged_to_concept_id, discharged_to_source_value, site_id, pregnancy_id)
      SELECT {own_visit_id}, infant_id, 9201, to_date(st), st, to_date(en), en, 32817,
             'SILVER-NNU', 0, dto, final_nnu_outcome, '{SITE}', pregnancy_id
      FROM {STG}._nnu_ep WHERE host_vid IS NULL""")
    spark.sql(f"""INSERT INTO {STG}.visit_detail (visit_detail_id, person_id, visit_detail_concept_id,
        visit_detail_start_date, visit_detail_start_datetime, visit_detail_end_date, visit_detail_end_datetime,
        visit_detail_type_concept_id, visit_detail_source_value, visit_detail_source_concept_id,
        discharged_to_concept_id, discharged_to_source_value, visit_occurrence_id, site_id, pregnancy_id, source_table)
      SELECT {silver_id('silver-nnu-detail', 'entity_id')}, infant_id, 9201, to_date(st), st, to_date(en), en, 32817,
             concat('NNU unit_level=', coalesce(unit_level, '~')), 0, dto, final_nnu_outcome,
             coalesce(host_vid, {own_visit_id}), '{SITE}', pregnancy_id, 'silver.clinical_neonatal_episode'
      FROM {STG}._nnu_ep""")


def add_neonatal_examinations():
    """Head circumference measurements and NIPE examination procedures from the episode's examinations."""
    exams = f"""SELECT x.patient_event_key AS xk, ne.infant_id, ne.pregnancy_id, ne.vid, ne.vdid,
        coalesce(x.examination_datetime, x.exam_date_derived) AS dt, x.*
      FROM {SILVER}.clinical_neonatal_examination x
      JOIN (SELECT n.*, coalesce(n.host_vid, {silver_id('silver-nnu-visit', 'n.entity_id')}) AS vid,
                   {silver_id('silver-nnu-detail', 'n.entity_id')} AS vdid FROM {STG}._nnu_ep n) ne
        ON ne.entity_id = x.entity_id
      WHERE x.record_status = 'active' AND coalesce(x.examination_datetime, x.exam_date_derived) IS NOT NULL"""
    low, high = HEAD_CIRCUMFERENCE_RANGE
    spark.sql(f"""INSERT INTO {STG}.measurement (measurement_id, person_id, measurement_concept_id, measurement_date,
        measurement_datetime, measurement_type_concept_id, value_as_number, unit_concept_id, unit_source_value,
        measurement_source_value, measurement_source_concept_id, visit_occurrence_id, visit_detail_id, site_id,
        pregnancy_id, source_table)
      SELECT {silver_id('silver-nnu-hc', 'e.xk')}, e.infant_id, 3000053, to_date(e.dt), e.dt, 32817,
             CAST(e.head_circumference AS DOUBLE), 8582, 'cm', 'NNU exam head circumference', 0, e.vid, e.vdid,
             '{SITE}', e.pregnancy_id, 'silver.clinical_neonatal_examination'
      FROM ({exams}) e WHERE e.head_circumference BETWEEN {low} AND {high}""")

    items = " UNION ALL ".join(
        f"SELECT xk, infant_id, pregnancy_id, vid, vdid, dt, '{label}' AS item, {concept} AS cid, {column} AS finding FROM ({exams})"
        for column, (label, concept) in NNU_EXAM_ITEMS.items())
    spark.sql(f"""INSERT INTO {STG}.procedure_occurrence (procedure_occurrence_id, person_id, procedure_concept_id,
        procedure_date, procedure_datetime, procedure_type_concept_id, procedure_source_value,
        procedure_source_concept_id, visit_occurrence_id, visit_detail_id, site_id, pregnancy_id, source_table)
      SELECT {silver_id('silver-nnu-exam', 'i.xk', 'i.item')}, i.infant_id, i.cid, to_date(i.dt), i.dt, 32817,
             concat('NIPE ', i.item, ': ', i.finding), 0, i.vid, i.vdid, '{SITE}', i.pregnancy_id,
             'silver.clinical_neonatal_examination'
      FROM ({items}) i WHERE nullif(trim(i.finding), '') IS NOT NULL AND lower(trim(i.finding)) <> 'not examined'""")


def add_neonatal_deaths():
    """A death for babies whose episode ended 'Died', where PREBORN has no death for them yet."""
    spark.sql(f"""INSERT INTO {STG}.death (person_id, death_date, death_datetime, death_type_concept_id, cause_concept_id,
        cause_source_value, cause_source_concept_id, site_id)
      SELECT infant_id, to_date(min(en)), min(en), 32817, 0, '{NNU_DEATH_SOURCE}', 0, '{SITE}'
      FROM {STG}._nnu_ep n WHERE final_nnu_outcome = 'Died'
        AND NOT EXISTS (SELECT 1 FROM {STG}.death d WHERE d.person_id = n.infant_id)
      GROUP BY infant_id""")


# The rows the neonatal lane adds: (table, id column, how to recognise them, provenance rule id).
NNU_ROWS = [
    ("visit_occurrence", "visit_occurrence_id", "visit_source_value = 'SILVER-NNU'", "SILVER-NNU-VISIT"),
    ("visit_detail", "visit_detail_id", "source_table = 'silver.clinical_neonatal_episode'", "SILVER-NNU-DETAIL"),
    ("measurement", "measurement_id", "source_table = 'silver.clinical_neonatal_examination'", "SILVER-NNU-HC"),
    ("procedure_occurrence", "procedure_occurrence_id", "source_table = 'silver.clinical_neonatal_examination'", "SILVER-NNU-EXAM"),
    ("death", "person_id", f"cause_source_value = '{NNU_DEATH_SOURCE}'", "SILVER-NNU-DEATH"),
]


def record_neonatal_provenance():
    """One _ehr_provenance row per neonatal-lane row; then check no table gained a duplicate id."""
    for table, id_column, is_nnu_row, rule_id in NNU_ROWS:
        spark.sql(f"""INSERT INTO {STG}._ehr_provenance (site_id, cdm_table, cdm_id, map_method, map_version, map_cosine, map_rule_id)
          SELECT '{SITE}', '{table}', {id_column}, 'silver_rule', CAST(NULL AS STRING), CAST(NULL AS DOUBLE), '{rule_id}'
          FROM {STG}.{table} WHERE site_id = '{SITE}' AND {is_nnu_row}""")
        assert_unique(f"{STG}.{table}", id_column, f"site_id = '{SITE}'")


def add_silver_neonatal_lane():
    """Run the neonatal lane and add its report rows to BUILD_REPORT."""
    route_conflicts = link_neonatal_episodes()
    stage_neonatal_episodes()
    add_neonatal_visits_and_stays()
    add_neonatal_examinations()
    add_neonatal_deaths()
    record_neonatal_provenance()

    r = spark.sql(f"""SELECT
        (SELECT count(*) FROM {STG}._nnu_link) AS linked,
        (SELECT count_if(method = 'mother_birth_time') FROM {STG}._nnu_link) AS via_mother,
        (SELECT count(*) FROM {STG}._nnu_ep) AS built,
        (SELECT count_if(host_vid IS NULL) FROM {STG}._nnu_ep) AS own_visit,
        (SELECT count(*) FROM {STG}._nnu_ep n JOIN {STG}.visit_detail v ON v.person_id = n.infant_id
           AND v.source_table = 'msd402neoadmiss' AND v.visit_detail_start_date = to_date(n.st)) AS overlap_402""").first()
    BUILD_REPORT.extend([
        ("silver NNU episodes linked", "report", True,
         f"{r.linked} ({r.via_mother} via mother+birth time); {r.built} built, {r.linked - r.built} skipped for no admit date"),
        ("silver NNU route conflicts left unlinked", "report", route_conflicts == 0, f"{route_conflicts} episodes"),
        ("silver NNU episodes needing their own visit", "report", True, f"{r.own_visit}"),
        ("silver NNU detail overlaps MSDS-402 same infant+day (both kept)", "report", True, f"{r.overlap_402}"),
    ])
    for row in BUILD_REPORT[-4:]:
        print(row)


add_silver_neonatal_lane()

## 7. Fills for empty MSDS values

MSDS stays authoritative, so pooled sites stay comparable: a silver value is written only where MSDS left the
column NULL. Where both exist and disagree, MSDS is kept and the disagreement is reported.

| Column | Silver source |
|---|---|
| marital status | `spine_person.marital_status_code` (Cerner code), through the linked mother |
| pre-pregnancy BMI | the earliest BMI on the pregnancy's VTE risk assessment form (12 to 80) |
| birth weight | the neonatal episode's birth weight (200 to 7000 g), neonatal unit babies only, when all their episodes agree |

Apgar, gravidity and parity have no silver source yet; that is reported.

In [0]:
# Cerner marital status code -> OMOP marital status concept.
MARITAL_CASE = """CASE {col}
  WHEN 309239 THEN 4053842
  WHEN 309237 THEN 4338692
  WHEN 309241 THEN 4143188
  WHEN 309236 THEN 4069297
  WHEN 309238 THEN 4027529
  ELSE 0 END"""
BMI_RANGE = (12, 80)
BIRTH_WEIGHT_RANGE = (200, 7000)


def fill_where_null(table, key, column, source_sql, source_column=None, source_value=None):
    """Set STG.table.column from source_sql (columns k = the key, v = the value) where it is NULL, and set
    source_column to source_value on the filled rows. Returns (filled, already equal, disagreeing)."""
    counts = spark.sql(f"""SELECT count_if(t.`{column}` IS NULL) AS filled,
        count_if(t.`{column}` = s.v) AS agree,
        count_if(t.`{column}` IS NOT NULL AND t.`{column}` <> s.v) AS disagree
      FROM {STG}.{table} t JOIN ({source_sql}) s ON s.k = t.`{key}`""").first()
    assignments = [f"t.`{column}` = s.v"]
    if source_column:
        assignments.append(f"t.`{source_column}` = {source_value}")
    spark.sql(f"""MERGE INTO {STG}.{table} t USING ({source_sql}) s ON s.k = t.`{key}`
      WHEN MATCHED AND t.`{column}` IS NULL THEN UPDATE SET {", ".join(assignments)}""")
    return counts.filled, counts.agree, counts.disagree


def add_silver_fills():
    """Fill marital status, pre-pregnancy BMI and birth weight where MSDS left them NULL; report the results."""
    pregnancy_id = f"CAST({mint_id('pregnancy', 'x.pregnancy_id')} AS BIGINT)"
    marital_concept = MARITAL_CASE.format(col="sp.marital_status_code")
    results = []

    marital = f"""SELECT p.pregnancy_id AS k, CAST({marital_concept} AS BIGINT) AS v,
        concat('spine_person:', cast(sp.marital_status_code AS STRING)) AS sv
      FROM {STG}.pregnancy p JOIN {STG}._bridge b ON b.person_id = p.person_id AND b.silver_person_id IS NOT NULL
      JOIN {SILVER}.spine_person sp ON sp.person_id = b.silver_person_id
      WHERE p.site_id = '{SITE}' AND {marital_concept} <> 0"""
    results.append(("pregnancy_marital_status_concept_id",
                    fill_where_null("pregnancy", "pregnancy_id", "pregnancy_marital_status_concept_id", marital,
                                    "pregnancy_marital_status_source_value", "s.sv")))

    bmi = f"""SELECT k, v FROM (
        SELECT {pregnancy_id} AS k, CAST(x.bmi AS DOUBLE) AS v, row_number() OVER (PARTITION BY x.pregnancy_id
               ORDER BY x.authored_datetime ASC NULLS LAST, x.patient_event_key ASC) AS rn
        FROM {SILVER}.clinical_form_maternity_vte_risk_assessment x
        WHERE x.record_status = 'active' AND x.pregnancy_id IS NOT NULL
          AND x.bmi BETWEEN {BMI_RANGE[0]} AND {BMI_RANGE[1]}) WHERE rn = 1"""
    results.append(("pre_pregnancy_bmi",
                     fill_where_null("pregnancy", "pregnancy_id", "pre_pregnancy_bmi", bmi,
                                     "pre_pregnancy_bmi_source", "'silver:clinical_form_maternity_vte_risk_assessment'")))

    birth_weight = f"""SELECT l.infant_id AS k, CAST(min(e.birthweight) AS DOUBLE) AS v
      FROM {STG}._nnu_link l JOIN {SILVER}.clinical_neonatal_episode e ON e.entity_id = l.entity_id
      WHERE e.birthweight BETWEEN {BIRTH_WEIGHT_RANGE[0]} AND {BIRTH_WEIGHT_RANGE[1]}
      GROUP BY l.infant_id HAVING count(DISTINCT e.birthweight) = 1"""
    results.append(("birth_weight",
                    fill_where_null("infant", "infant_id", "birth_weight", birth_weight,
                                    "birth_weight_source", "'silver:clinical_neonatal_episode'")))

    for column, (filled, agree, disagree) in results:
        BUILD_REPORT.append((f"silver fill {column}", "report", True, f"filled {filled}, already MSDS-equal {agree}"))
    for column, (filled, agree, disagree) in results:
        BUILD_REPORT.append((f"concordance {column} (MSDS kept)", "report", disagree == 0,
                             f"{disagree} MSDS values disagree with silver"))
    for column in ("birth_apgar", "prev_pregnancy_gravidity", "prev_pregnancy_parity_concept_id"):
        BUILD_REPORT.append((f"silver fill {column}", "report", False,
                             "no silver source (clinical_birth / pregnancy history not landed)"))
    for column, counts in results:
        print(column, "filled / already equal / disagree:", counts)


add_silver_fills()

## 8. Maternity-relevant EHR facts

The EHR holds far more than a maternity study needs, so only facts whose concept is in a **maternity concept
set** are taken, and only inside each person's **window**:

- mothers: from 365 days before each pregnancy's start to 365 days after its end;
- babies: from birth to 730 days after birth.

A pregnancy with only a start or only an end is given the other as 294 days (42 weeks) away.

### 8a. The maternity concept set

Built from **anchor** concepts (whole SNOMED hierarchies, such as 'Disorder of pregnancy') and named **groups**
of measurements and drugs, each tagged with the role (mother or infant) it applies to. Each entry is expanded to
all its standard descendants, plus the non-standard concepts that 'Map to' them (silver keeps some drug concepts
as dm+d or SNOMED substances). The set is published as `_concept_set`, with the anchors and groups that admitted
each concept.

Every entry is first checked against the live vocabulary on code, exact name, domain and standard flag: a code
that now means something else must stop the build, not quietly admit the wrong hierarchy.

In [0]:
MATERNITY_ANCHORS = [  # (vocab, code, exact concept_name, domain, group, role)
    ("SNOMED", "77386006", "Pregnancy", "Condition", "pregnancy", "mother"),
    ("SNOMED", "118185001", "Finding related to pregnancy", "Condition", "pregnancy", "mother"),
    ("SNOMED", "173300003", "Disorder of pregnancy", "Condition", "pregnancy", "mother"),
    ("SNOMED", "364320009", "Pregnancy observable", "Observation", "pregnancy", "mother"),
    ("SNOMED", "118216002", "Labor finding", "Condition", "labour", "mother"),
    ("SNOMED", "362973001", "Disorder of puerperium", "Condition", "puerperium", "mother"),
    ("SNOMED", "386637004", "Obstetric procedure", "Procedure", "obstetric_procedures", "mother"),
    ("SNOMED", "414025005", "Disorder of fetus and/or newborn", "Condition", "fetus_newborn", "infant"),
    ("SNOMED", "118188004", "Finding of neonate", "Condition", "fetus_newborn", "infant"),
]
# Named groups: (vocab, code, exact concept_name), expanded to their standard descendants. Codes were picked from
# the concepts silver actually carries for linked PREBORN people (2026-10-01). ATC entries are classification
# concepts (standard 'C') whose descendants are the RxNorm drugs; measurement entries are SNOMED or LOINC.
VITAL_SIGNS = [("SNOMED", "271649006", "Systolic blood pressure"), ("SNOMED", "271650006", "Diastolic blood pressure"),
           ("SNOMED", "27113001", "Body weight"), ("SNOMED", "50373000", "Body height measure"),
           ("SNOMED", "60621009", "Body mass index"), ("SNOMED", "386725007", "Body temperature"),
           ("SNOMED", "78564009", "Pulse rate")]
ANTIBIOTICS = [("ATC", "J01CE01", "benzylpenicillin; parenteral"), ("ATC", "J01CA04", "amoxicillin; oral, parenteral"),
                ("ATC", "J01CR02", "amoxicillin and beta-lactamase inhibitor; oral, parenteral"),
                ("ATC", "J01DC02", "cefuroxime; oral, parenteral"), ("ATC", "J01FF01", "clindamycin; oral, parenteral"),
                ("ATC", "J01FA01", "erythromycin; oral, parenteral"), ("ATC", "J01GB03", "gentamicin; parenteral"),
                ("ATC", "J01XD01", "metronidazole; parenteral"), ("ATC", "P01AB01", "metronidazole; oral, rectal")]
MATERNITY_GROUPS = {  # group -> (roles, entries)
    "maternal_vitals": (("mother",), VITAL_SIGNS),
    "antenatal_bloods": (("mother",), [
        ("SNOMED", "313995005", "Hemoglobin A measurement"), ("SNOMED", "441689006", "Measurement of total hemoglobin concentration"),
        ("SNOMED", "489004", "Ferritin measurement"), ("SNOMED", "61928009", "Platelet count"),
        ("SNOMED", "33747003", "Glucose measurement, blood"), ("SNOMED", "72191006", "Glucose measurement, plasma"),
        ("SNOMED", "43396009", "Hemoglobin A1c measurement"), ("SNOMED", "113076002", "Glucose tolerance test"),
        ("SNOMED", "44608003", "Blood group typing"), ("SNOMED", "120646007", "Antibody screen"),
        ("SNOMED", "89754000", "RBC antibody detection"), ("SNOMED", "395059005", "Hemoglobinopathy screening test")]),
    "antenatal_infection_screen": (("mother",), [
        ("SNOMED", "165813002", "Human immunodeficiency virus antibody test"),
        ("SNOMED", "1020841000000100", "HIV (human immunodeficiency virus) serology"),
        ("SNOMED", "50421000237109", "HIV (human immunodeficiency virus) type 2 antibody in serum qualitative result"),
        ("SNOMED", "117750009", "HIV 1 p24 antigen assay"),
        ("LOINC", "95524-5", "HIV 1 and 2 Ab [Identifier] in Serum or Plasma by Immunoassay"),
        ("SNOMED", "47758006", "Hepatitis B surface antigen measurement"),
        ("SNOMED", "374971000119102", "Treponema pallidum antibody by screening enzyme immunoassay"),
        ("SNOMED", "313670007", "Rubella IgG measurement"), ("SNOMED", "9954002", "Serologic test for rubella"),
        ("SNOMED", "50981000237103", "Rubella virus total antibody in serum qualitative result")]),
    "pre_eclampsia_markers": (("mother",), [
        ("SNOMED", "57378007", "Urine protein measurement"), ("SNOMED", "250750009", "Protein/creatinine ratio measurement"),
        ("SNOMED", "889391000000100", "Alanine transaminase activity measurement"),
        ("SNOMED", "364571000119101", "Alanine transaminase in serum"),
        ("SNOMED", "313936008", "Plasma creatinine measurement"), ("SNOMED", "113075003", "Creatinine measurement, serum"),
        ("SNOMED", "313822004", "Corrected serum creatinine measurement"), ("SNOMED", "270984004", "Blood urate measurement"),
        ("LOINC", "74755-0", "Placental growth factor [Mass/volume] in Serum"),
        ("LOINC", "74756-8", "Soluble fms-like tyrosine kinase-1 [Mass/volume] in Serum"),
        ("LOINC", "74757-6", "Soluble fms-like tyrosine kinase-1/placental growth factor [Ratio] in Serum")]),
    "thyroid": (("mother",), [("SNOMED", "61167004", "Thyroid stimulating hormone measurement"),
                              ("SNOMED", "5113004", "T4 free measurement")]),
    "infant_growth": (("infant",), [
        ("SNOMED", "27113001", "Body weight"), ("SNOMED", "50373000", "Body height measure"),
        ("SNOMED", "363812007", "Head circumference"), ("LOINC", "9843-4", "Head Occipital-frontal circumference")]),
    "neonatal_bilirubin": (("infant",), [("SNOMED", "302787001", "Bilirubin measurement")]),
    "newborn_blood_spot": (("infant",), [
        ("SNOMED", "428447008", "Newborn blood spot screening"), ("SNOMED", "314081000", "Phenylketonuria screening test"),
        ("SNOMED", "400984005", "Congenital hypothyroidism screening test"),
        ("SNOMED", "314090007", "Sickle cell disease screening test"),
        ("SNOMED", "1015871000000108", "Cystic fibrosis screening test"),
        ("SNOMED", "428056008", "Medium-chain acyl-coenzyme A dehydrogenase deficiency screening test")]),
    "maternity_drugs": (("mother",), [
        ("ATC", "H01BB02", "oxytocin; nasal, oral, parenteral"), ("ATC", "G02AD04", "carboprost; parenteral"),
        ("ATC", "G02AB03", "ergometrine; oral, parenteral"), ("ATC", "G02AC01", "methylergometrine and oxytocin"),
        ("ATC", "A02BB01", "misoprostol; oral"), ("ATC", "G02AD06", "misoprostol; oral, vaginal"),
        ("ATC", "G02CX01", "atosiban; parenteral"),
        ("ATC", "H02AB01", "betamethasone; oral, parenteral"), ("ATC", "H02AB02", "dexamethasone; oral, parenteral"),
        ("ATC", "A12CC02", "magnesium sulfate; oral, parenteral"), ("ATC", "B05XA05", "magnesium sulfate"),
        ("ATC", "C07AG01", "labetalol; oral, parenteral"), ("ATC", "C08CA05", "nifedipine; oral, parenteral"),
        ("ATC", "C02AB01", "methyldopa (levorotatory); oral, parenteral"),
        ("ATC", "B01AC06", "acetylsalicylic acid; oral"), ("ATC", "B03BB01", "folic acid; oral, parenteral"),
        ("ATC", "B03A", "IRON PREPARATIONS"), ("ATC", "A10A", "INSULINS AND ANALOGUES"), ("ATC", "A10BA02", "metformin; oral"),
        ("ATC", "B01AB05", "enoxaparin; parenteral"), ("ATC", "B01AB04", "dalteparin; parenteral"),
        ("ATC", "B01AB10", "tinzaparin; parenteral"), ("ATC", "J06BB01", "anti-D (rh) immunoglobulin")]),
    "peripartum_antibiotics": (("mother", "infant"), ANTIBIOTICS),   # intrapartum prophylaxis; neonatal sepsis
    "infant_drugs": (("infant",), [("ATC", "B02BA01", "phytomenadione; oral, parenteral"), ("ATC", "J07", "VACCINES")]),
}

# Days before and after: mothers around each pregnancy, babies after birth.
WINDOW_DAYS = {"mother": (365, 365), "infant": (0, 730)}


def build_maternity_concept_set():
    """Create STG._concept_set: one row per (concept, role) admitted by MATERNITY_ANCHORS or MATERNITY_GROUPS."""
    entries = [(vocab, code, name, domain, group, role) for vocab, code, name, domain, group, role in MATERNITY_ANCHORS]
    for group, (roles, members) in MATERNITY_GROUPS.items():
        for vocab, code, name in members:
            domain = "Drug" if vocab == "ATC" else "Measurement"
            for role in roles:
                entries.append((vocab, code, name, domain, group, role))
    values = ", ".join("(" + ", ".join(f"'{sql_str(x)}'" for x in entry) + ")" for entry in entries)
    seed = f"SELECT * FROM VALUES {values} AS t(vocab, code, name, domain, grp, role)"

    # Each entry must still be the concept we chose.
    bad = spark.sql(f"""SELECT k.vocab, k.code, k.name, c.concept_name, c.domain_id, c.standard_concept
      FROM ({seed}) k LEFT JOIN {VOCAB}.concept c ON c.vocabulary_id = k.vocab AND c.concept_code = k.code
      WHERE c.concept_id IS NULL OR c.concept_name <> k.name OR c.domain_id <> k.domain
         OR c.standard_concept IS DISTINCT FROM CASE WHEN k.vocab = 'ATC' THEN 'C' ELSE 'S' END""").collect()
    if bad:
        problems = [f"{r.vocab}:{r.code} expected {r.name!r}, found {r.concept_name!r} {r.domain_id}/{r.standard_concept}"
                    for r in bad]
        raise AssertionError("maternity concept pins no longer hold: " + "; ".join(problems))

    spark.sql(f"""INSERT INTO {STG}._concept_set (site_id, concept_id, role, anchors, groups)
      WITH k AS (SELECT k.*, c.concept_id AS root FROM ({seed}) k
                 JOIN {VOCAB}.concept c ON c.vocabulary_id = k.vocab AND c.concept_code = k.code),
      -- the standard descendants of each entry
      d AS (SELECT a.descendant_concept_id AS concept_id, k.role, concat(k.vocab, ':', k.code) AS anchor, k.grp
            FROM k JOIN {VOCAB}.concept_ancestor a ON a.ancestor_concept_id = k.root
            JOIN {VOCAB}.concept c ON c.concept_id = a.descendant_concept_id AND c.standard_concept = 'S'),
      -- the concepts that map to them
      m AS (SELECT r.concept_id_1 AS concept_id, d.role, d.anchor, d.grp
            FROM d JOIN {VOCAB}.concept_relationship r ON r.concept_id_2 = d.concept_id
             AND r.relationship_id = 'Maps to' AND r.invalid_reason IS NULL AND r.concept_id_1 <> r.concept_id_2)
      SELECT '{SITE}', concept_id, role, array_sort(collect_set(anchor)), array_sort(collect_set(grp))
      FROM (SELECT * FROM d UNION ALL SELECT * FROM m) GROUP BY concept_id, role""")
    rows, keys = spark.sql(f"SELECT count(*), count(DISTINCT concept_id, role) FROM {STG}._concept_set").first()
    assert rows == keys, f"_concept_set is not one row per (concept_id, role): {rows} rows, {keys} keys"
    for r in spark.sql(f"SELECT role, count(*) n FROM {STG}._concept_set GROUP BY role ORDER BY role").collect():
        BUILD_REPORT.append((f"concept set {r.role}", "report", True, f"{r.n} concepts"))
        print("concept set", r.role, r.n)


# Each pregnancy's span; a missing start or end is taken as 294 days (42 weeks) from the other.
PREGNANCY_SPANS = f"""SELECT person_id, pregnancy_id,
      coalesce(pregnancy_start_date, date_sub(pregnancy_end_date, 294)) AS ps,
      coalesce(pregnancy_end_date, date_add(pregnancy_start_date, 294)) AS pe
    FROM {STG}.pregnancy WHERE site_id = '{SITE}' AND coalesce(pregnancy_start_date, pregnancy_end_date) IS NOT NULL"""


def build_windows():
    """Create STG._window(site_id, person_id, role, from_date, to_date): one row per mother's pregnancy and one
    per baby. Rows may overlap; facts are matched with EXISTS, so an overlap never copies a fact."""
    (mother_before, mother_after), (infant_before, infant_after) = WINDOW_DAYS["mother"], WINDOW_DAYS["infant"]
    spark.sql(f"""CREATE OR REPLACE TABLE {STG}._window AS
      SELECT '{SITE}' AS site_id, person_id, 'mother' AS role,
             date_sub(ps, {mother_before}) AS from_date, date_add(pe, {mother_after}) AS to_date
      FROM ({PREGNANCY_SPANS})
      UNION ALL
      SELECT '{SITE}', p.person_id, 'infant',
             date_sub(to_date(p.birth_datetime), {infant_before}), date_add(to_date(p.birth_datetime), {infant_after})
      FROM {STG}.infant i JOIN {STG}.person p ON p.person_id = i.infant_id AND p.site_id = '{SITE}'
      WHERE i.site_id = '{SITE}' AND p.birth_datetime IS NOT NULL""")


build_maternity_concept_set()
build_windows()

### 8b. The EHR facts

For each silver source in `EHR_SPECS`: current rows of linked people, inside their window, with a concept in the
set for their role. Each fact keeps silver's mapping provenance (method, version, cosine, rule) in
`_ehr_provenance`, and is assigned:

- a **pregnancy**: a baby's own, or the mother's pregnancy whose span (start to end + 42 days) holds the date,
  the latest-starting first;
- a **visit**: its silver encounter, if that encounter is kept.

**Encounters** are kept when a kept fact points at them, or when they are a mother's and start inside one of
her pregnancies. Waiting-list entries and other non-visits are not encounters; a fact pointing at one loses its
visit link.

**Deaths**: silver death evidence is added only for people PREBORN has no death for (MSDS, then the neonatal
unit, win); disagreements of more than a day are reported.

The lane stops before writing if any table would receive more than 50 million rows.

In [0]:
# Given doses only: the OGR prod router's rule (In Error 31, Not Done 36, Not Given / Not Done displays).
ADMINISTERED_ONLY = """coalesce(try_cast(administration_status_code AS INT), -1) NOT IN (31, 36)
  AND lower(coalesce(source_event_type_display, '')) NOT IN ('not given', 'not done')"""
# One entry per silver source:
#   table  the OMOP table it goes to;   id  that table's id column;   grain  the id kind for mint_id
#   src    the silver table;            cid the silver concept column (checked against the concept set)
#   dt     the event time;              where  which silver rows count;   prov  the prefix of silver's *_map_* columns
#   cols   OMOP column -> SQL over the silver row (bare silver column names; _dt is the event time)
EHR_SPECS = [
    dict(table="condition_occurrence", id="condition_occurrence_id", grain="ehr-condition", src="clinical_condition",
         cid="condition_omop_concept_id", dt="coalesce(onset_datetime, event_datetime)", prov="condition",
         where="coalesce(clinical_status_display, '') <> 'Canceled' AND coalesce(verification_status_display, '') <> 'Excluded'",
         cols={"condition_concept_id": "condition_omop_concept_id", "condition_start_date": "to_date(_dt)",
               "condition_start_datetime": "_dt", "condition_end_date": "to_date(abatement_datetime)",
               "condition_end_datetime": "abatement_datetime", "condition_type_concept_id": "32817",
               "condition_status_concept_id": "0", "condition_status_source_value": "verification_status_display",
               "condition_source_concept_id": "coalesce(condition_omop_source_concept_id, 0)"}),
    dict(table="procedure_occurrence", id="procedure_occurrence_id", grain="ehr-procedure", src="clinical_procedure",
         cid="procedure_omop_concept_id", dt="coalesce(performed_start, event_datetime)", prov="procedure",
         where="coalesce(status_display, '') NOT IN ('SCHEDULED_ONLY', 'CANCELLED')",
         cols={"procedure_concept_id": "procedure_omop_concept_id", "procedure_date": "to_date(_dt)",
               "procedure_datetime": "_dt", "procedure_end_date": "to_date(performed_end)",
               "procedure_end_datetime": "performed_end", "procedure_type_concept_id": "32817",
               "quantity": "CAST(quantity AS BIGINT)",
               "procedure_source_concept_id": "coalesce(procedure_omop_source_concept_id, 0)"}),
    dict(table="drug_exposure", id="drug_exposure_id", grain="ehr-drug-admin", src="clinical_medication_admin",
         cid="medication_omop_concept_id", dt="coalesce(performed_datetime, event_datetime)", prov="medication",
         where=ADMINISTERED_ONLY,
         cols={"drug_concept_id": "medication_omop_concept_id", "drug_exposure_start_date": "to_date(_dt)",
               "drug_exposure_start_datetime": "_dt", "drug_exposure_end_date": "to_date(_dt)",
               "drug_exposure_end_datetime": "_dt", "drug_type_concept_id": "32817",
               "quantity": "CAST(dose_value AS DOUBLE)", "route_concept_id": "0", "route_source_value": "route_display",
               "dose_unit_source_value": "dose_unit",
               "drug_source_concept_id": "coalesce(medication_omop_source_concept_id, 0)"}),
    dict(table="measurement", id="measurement_id", grain="ehr-pathology", src="clinical_pathology_result",
         cid="test_omop_concept_id", dt="event_datetime", prov="test",
         where="coalesce(preferred_result_ind, true) AND person_projection_status = 'eligible'",
         cols={"measurement_concept_id": "test_omop_concept_id", "measurement_date": "to_date(_dt)",
               "measurement_datetime": "_dt", "measurement_type_concept_id": "32817",
               "operator_concept_id": "coalesce(try_cast(operator_concept_id AS BIGINT), 0)",
               "value_as_number": "CAST(value_number AS DOUBLE)",
               "value_as_concept_id": "coalesce(try_cast(value_concept_id AS BIGINT), 0)",
               "unit_concept_id": "coalesce(try_cast(unit_concept_id AS BIGINT), 0)",
               "unit_source_value": "unit_source_value", "range_low": "CAST(reference_range_low AS DOUBLE)",
               "range_high": "CAST(reference_range_high AS DOUBLE)", "value_source_value": "left(value_text, 250)",
               "measurement_source_concept_id": "coalesce(test_omop_source_concept_id, 0)"}),
    dict(table="measurement", id="measurement_id", grain="ehr-vital", src="clinical_vital_sign",
         cid="vital_omop_concept_id", dt="event_datetime", prov="vital", where="true",
         cols={"measurement_concept_id": "vital_omop_concept_id", "measurement_date": "to_date(_dt)",
               "measurement_datetime": "_dt", "measurement_type_concept_id": "32817",
               "value_as_number": "CAST(value_number AS DOUBLE)", "value_as_concept_id": "0",
               "unit_concept_id": "coalesce(unit_omop_concept_id, 0)", "unit_source_value": "unit_source_value",
               "range_low": "CAST(reference_range_low AS DOUBLE)", "range_high": "CAST(reference_range_high AS DOUBLE)",
               "value_source_value": "left(value_text, 250)",
               "measurement_source_concept_id": "coalesce(vital_omop_source_concept_id, 0)"}),
    dict(table="observation", id="observation_id", grain="ehr-allergy", src="clinical_allergy_intolerance",
         cid="substance_omop_concept_id", dt="event_datetime", prov="substance",
         where="NOT coalesce(absence_assertion_ind, false)",
         cols={"observation_concept_id": "439224", "observation_date": "to_date(_dt)", "observation_datetime": "_dt",
               "observation_type_concept_id": "32817", "value_as_concept_id": "substance_omop_concept_id",
               "value_as_string": "reaction_status_display",
               "observation_source_concept_id": "coalesce(substance_omop_source_concept_id, 0)"}),
    dict(table="device_exposure", id="device_exposure_id", grain="ehr-device", src="clinical_device",
         cid="device_omop_concept_id", dt="coalesce(implanted_datetime, event_datetime)", prov="device", where="true",
         cols={"device_concept_id": "device_omop_concept_id", "device_exposure_start_date": "to_date(_dt)",
               "device_exposure_start_datetime": "_dt", "device_type_concept_id": "32817",
               "device_source_concept_id": "coalesce(device_omop_source_concept_id, 0)"}),
]
SOURCE_VALUE_COLUMN = {"condition_occurrence": "condition_source_value", "procedure_occurrence": "procedure_source_value",
                  "drug_exposure": "drug_source_value", "measurement": "measurement_source_value",
                  "observation": "observation_source_value", "device_exposure": "device_source_value"}
EHR_TABLES = sorted({sp["table"] for sp in EHR_SPECS})
EHR_SOURCE_TABLES = sorted({"silver." + sp["src"] for sp in EHR_SPECS})
EHR_DEATH_SOURCE = "silver:reference_person_death_evidence"
# Visit concept from spine_encounter.encounter_level; waiting-list placeholders and 'other' are not visits.
EHR_VISIT_CASE = """CASE encounter_level WHEN 'spell' THEN 9201 WHEN 'emergency_visit' THEN 9203
  WHEN 'outpatient_attendance' THEN 9202 WHEN 'preadmission' THEN 9202 WHEN 'results_only' THEN 32036
  WHEN 'recurring_contact' THEN CASE WHEN type_display = 'Emergency Department' THEN 9203 ELSE 9202 END END"""
EHR_MAX_ROWS = 50_000_000


def ehr_scope_sql(spec, in_concept_set=True):
    """SELECT over one silver source: current rows of linked people inside their window (adds _pid, _role, _dt).
    With in_concept_set, only rows whose concept is in the maternity set for the person's role."""
    concept_set_filter = ""
    if in_concept_set:
        concept_set_filter = f"""AND EXISTS (SELECT 1 FROM {STG}._concept_set c WHERE c.site_id = '{SITE}'
               AND c.concept_id = y.{spec['cid']} AND c.role = y._role)"""
    return f"""SELECT y.*, {spec['dt']} AS _dt FROM (
        SELECT x.*, b.person_id AS _pid, b.role AS _role
        FROM {SILVER}.{spec['src']} x JOIN {STG}._bridge b ON b.silver_person_id = x.person_id
        WHERE x.record_status = 'active') y
      WHERE ({spec['where']}) AND {spec['dt']} IS NOT NULL
        AND EXISTS (SELECT 1 FROM {STG}._window w WHERE w.site_id = '{SITE}' AND w.person_id = y._pid
                    AND to_date({spec['dt']}) BETWEEN w.from_date AND w.to_date) {concept_set_filter}"""


def pregnancy_join_sql(alias):
    """LEFT JOINs giving a row (alias, with _role, _pid, _dt) its pregnancy: i = the baby's own, p = the mother's
    pregnancy whose span (start to end + 42 days) holds the date."""
    return f"""LEFT JOIN {STG}.infant i ON {alias}._role = 'infant' AND i.infant_id = {alias}._pid AND i.site_id = '{SITE}'
      LEFT JOIN ({PREGNANCY_SPANS}) p ON {alias}._role = 'mother' AND p.person_id = {alias}._pid
       AND to_date({alias}._dt) BETWEEN p.ps AND date_add(p.pe, 42)"""


def ehr_volumes():
    """Per source: rows in window, how many are unmapped (concept 0 or NULL), and how many the concept set keeps."""
    parts = []
    for spec in EHR_SPECS:
        # _concept_set is one row per (concept_id, role), so this join cannot multiply rows.
        parts.append(f"""SELECT '{spec['table']}' AS cdm_table, '{spec['src']}' AS src, count(*) AS in_window,
              count_if(coalesce({spec['cid']}, 0) = 0) AS unmapped, count(c.concept_id) AS kept
            FROM ({ehr_scope_sql(spec, in_concept_set=False)}) z
            LEFT JOIN {STG}._concept_set c ON c.site_id = '{SITE}' AND c.concept_id = z.{spec['cid']} AND c.role = z._role""")
    return spark.sql(" UNION ALL ".join(parts)).collect()


def add_ehr_facts():
    """Insert each EHR_SPECS source's kept facts into its OMOP table, with one provenance row per fact."""
    for spec in EHR_SPECS:
        table = spec["table"]
        columns = dict(spec["cols"])
        columns[SOURCE_VALUE_COLUMN[table]] = f"concat('ehr:{spec['src']}:', coalesce(source_code, ''))"
        names = list(columns)
        prov = spec["prov"]
        spark.sql(f"""CREATE OR REPLACE TABLE {STG}._ehr_stage AS
          SELECT {silver_id(spec['grain'], 'f.patient_event_key')} AS _id, f._pid AS person_id,
                 {', '.join(f'{expression} AS {column}' for column, expression in columns.items())},
                 CASE WHEN f.encounter_id IS NOT NULL THEN {silver_id('ehr-visit', 'f.encounter_id')} END AS visit_occurrence_id,
                 coalesce(i.pregnancy_id, p.pregnancy_id) AS pregnancy_id,
                 f.{prov}_map_method AS _mm, f.{prov}_map_version AS _mv,
                 CAST(f.{prov}_map_cosine AS DOUBLE) AS _mc, f.{prov}_map_rule_id AS _mr
          FROM ({ehr_scope_sql(spec)}) f {pregnancy_join_sql('f')}
          QUALIFY row_number() OVER (PARTITION BY f.patient_event_key
                                     ORDER BY p.ps DESC NULLS LAST, p.pregnancy_id ASC NULLS LAST) = 1""")
        spark.sql(f"""INSERT INTO {STG}.{table} ({spec['id']}, person_id, {', '.join(names)}, visit_occurrence_id,
            site_id, pregnancy_id, source_table)
          SELECT _id, person_id, {', '.join(names)}, visit_occurrence_id, '{SITE}', pregnancy_id, 'silver.{spec['src']}'
          FROM {STG}._ehr_stage""")
        spark.sql(f"""INSERT INTO {STG}._ehr_provenance (site_id, cdm_table, cdm_id, map_method, map_version, map_cosine, map_rule_id)
          SELECT '{SITE}', '{table}', _id, _mm, _mv, _mc, _mr FROM {STG}._ehr_stage""")
        print("ehr facts:", spec["src"], "->", table)


EHR_SOURCES_IN = ", ".join(f"'{t}'" for t in EHR_SOURCE_TABLES)   # SQL list of the silver.* source_table values


def add_ehr_encounters():
    """Insert the kept encounters as visits, then remove visit links that point at an encounter not kept.
    Returns how many fact visit links were removed."""
    referenced = " UNION ".join(f"""SELECT visit_occurrence_id AS vid FROM {STG}.{t}
        WHERE site_id = '{SITE}' AND source_table IN ({EHR_SOURCES_IN}) AND visit_occurrence_id IS NOT NULL"""
                                for t in EHR_TABLES)
    spark.sql(f"""CREATE OR REPLACE TABLE {STG}._ehr_stage AS
      WITH e AS (
        SELECT x.*, {silver_id('ehr-visit', 'x.encounter_id')} AS vid, b.person_id AS _pid, b.role AS _role,
               coalesce(x.registration_datetime, x.period_start) AS _dt,
               coalesce(x.discharge_datetime, x.period_end) AS _end, {EHR_VISIT_CASE} AS vcid
        FROM {SILVER}.spine_encounter x JOIN {STG}._bridge b ON b.silver_person_id = x.person_id
        WHERE x.record_status = 'active')
      SELECT e.vid, e._pid, e.vcid, e._dt, CASE WHEN e._end >= e._dt THEN e._end ELSE e._dt END AS _end,
             e.type_display, e.encounter_level, e.admission_source_display, e.discharge_destination_display,
             coalesce(i.pregnancy_id, p.pregnancy_id) AS pregnancy_id
      FROM e {pregnancy_join_sql('e')}
      WHERE e.vcid IS NOT NULL AND e._dt IS NOT NULL
        AND (e.vid IN ({referenced}) OR (e._role = 'mother' AND p.pregnancy_id IS NOT NULL))
      QUALIFY row_number() OVER (PARTITION BY e.vid ORDER BY p.ps DESC NULLS LAST, p.pregnancy_id ASC NULLS LAST) = 1""")
    spark.sql(f"""INSERT INTO {STG}.visit_occurrence (visit_occurrence_id, person_id, visit_concept_id, visit_start_date,
        visit_start_datetime, visit_end_date, visit_end_datetime, visit_type_concept_id, visit_source_value,
        visit_source_concept_id, admitted_from_concept_id, admitted_from_source_value, discharged_to_concept_id,
        discharged_to_source_value, site_id, pregnancy_id)
      SELECT vid, _pid, vcid, to_date(_dt), _dt, to_date(_end), _end, 32817,
             concat('ehr:spine_encounter:', coalesce(type_display, encounter_level)), 0, 0, admission_source_display,
             0, discharge_destination_display, '{SITE}', pregnancy_id
      FROM {STG}._ehr_stage""")

    removed = 0
    for table in EHR_TABLES:
        result = spark.sql(f"""MERGE INTO {STG}.{table} f USING (
            SELECT DISTINCT visit_occurrence_id AS vid FROM {STG}.{table} x
            WHERE x.site_id = '{SITE}' AND x.source_table IN ({EHR_SOURCES_IN}) AND x.visit_occurrence_id IS NOT NULL
              AND NOT EXISTS (SELECT 1 FROM {STG}.visit_occurrence v WHERE v.visit_occurrence_id = x.visit_occurrence_id)) d
          ON f.visit_occurrence_id = d.vid AND f.site_id = '{SITE}' AND f.source_table IN ({EHR_SOURCES_IN})
          WHEN MATCHED THEN UPDATE SET visit_occurrence_id = NULL""").first()
        removed += result.num_updated_rows
    return removed


def add_ehr_deaths():
    """Add silver death evidence for people with no PREBORN death. Returns (people with both, disagreeing)."""
    spark.sql(f"""INSERT INTO {STG}.death (person_id, death_date, death_datetime, death_type_concept_id, cause_concept_id,
        cause_source_value, cause_source_concept_id, site_id)
      SELECT b.person_id, to_date(x.deceased_datetime), x.deceased_datetime, 32817, 0, '{EHR_DEATH_SOURCE}', 0, '{SITE}'
      FROM {SILVER}.reference_person_death_evidence x JOIN {STG}._bridge b ON b.silver_person_id = x.person_id
      WHERE x.record_status = 'active' AND x.deceased_datetime IS NOT NULL
        AND NOT EXISTS (SELECT 1 FROM {STG}.death d WHERE d.person_id = b.person_id AND d.site_id = '{SITE}')""")
    return spark.sql(f"""SELECT count(*) AS both, count_if(abs(datediff(d.death_date, to_date(x.deceased_datetime))) > 1) AS disagree
      FROM {STG}.death d JOIN {STG}._bridge b ON b.person_id = d.person_id
      JOIN {SILVER}.reference_person_death_evidence x ON x.person_id = b.silver_person_id
       AND x.record_status = 'active' AND x.deceased_datetime IS NOT NULL
      WHERE d.site_id = '{SITE}' AND d.cause_source_value IS DISTINCT FROM '{EHR_DEATH_SOURCE}'""").first()


def add_silver_ehr_lane():
    """Run the EHR lane and add its report rows to BUILD_REPORT."""
    volumes = ehr_volumes()
    kept_per_table = {}
    for v in volumes:
        kept_per_table[v.cdm_table] = kept_per_table.get(v.cdm_table, 0) + v.kept
    too_big = {table: n for table, n in kept_per_table.items() if n > EHR_MAX_ROWS}
    assert not too_big, f"EHR lane volume over {EHR_MAX_ROWS:,} rows: {too_big} -- agree the ceiling before inserting"

    add_ehr_facts()
    links_removed = add_ehr_encounters()
    deaths = add_ehr_deaths()

    checked = [("visit_occurrence", "visit_occurrence_id"), ("death", "person_id")]
    checked += [(spec["table"], spec["id"]) for spec in EHR_SPECS]
    for table, id_column in checked:
        assert_unique(f"{STG}.{table}", id_column, f"site_id = '{SITE}'")
    spark.sql(f"DROP TABLE IF EXISTS {STG}._ehr_stage")

    for v in volumes:
        BUILD_REPORT.append((f"ehr {v.src} -> {v.cdm_table}", "report", True, f"{v.kept} kept of {v.in_window} in window"))
    for v in volumes:
        BUILD_REPORT.append((f"ehr_unmapped_outside_set {v.src}", "report", v.unmapped == 0,
                             f"{v.unmapped} of {v.in_window} in-window rows unmapped (concept 0)"))
    visits = scalar(f"SELECT count(*) FROM {STG}.visit_occurrence WHERE site_id = '{SITE}' AND visit_source_value LIKE 'ehr:%'")
    added_deaths = scalar(f"SELECT count(*) FROM {STG}.death WHERE site_id = '{SITE}' AND cause_source_value = '{EHR_DEATH_SOURCE}'")
    BUILD_REPORT.extend([
        ("ehr visits", "report", True, f"{visits} encounters ({links_removed} fact visit links nulled: encounter not kept)"),
        ("ehr deaths added (no PREBORN death)", "report", True, f"{added_deaths}"),
        ("death concordance PREBORN vs silver evidence", "report", deaths.disagree == 0,
         f"{deaths.disagree} of {deaths.both} differ by more than a day (PREBORN kept)"),
        ("ehr medication dispense", "report", False, "not used: silver dispense rows carry no mapped concept"),
    ])
    for v in volumes:
        print(f"{v.src} -> {v.cdm_table}: {v.kept} kept of {v.in_window} in window ({v.unmapped} unmapped)")


add_silver_ehr_lane()

## 9. Cleaning

Only the rows this notebook added are cleaned; the seeded MSDS rows were cleaned by `10_preborn_msds` and are
never touched. Added rows are recognised by markers MSDS never writes (`BEYOND_ROWS`), never by "anything
that is not MSDS".

An added row is dropped, and recorded in `STG._dropped` as `BEYOND:<table>`, when, first match wins:

| Reason | Meaning |
|---|---|
| before_1900, after_release | outside the dataset's time window |
| before_birth | before the person was born (babies: more than a day before) |
| after_death | more than 30 days after the person's death (not applied to deaths) |
| parent_visit_dropped | a visit detail whose visit was dropped |

A surviving added fact whose visit or visit detail was dropped keeps its row, with that link set to NULL
(recorded as `FK:<table>.<column>`).

In [0]:
# The lanes' report rows are kept with the working tables, so the QA log (section 11) can include them.
spark.createDataFrame([(name, severity, bool(ok), str(metric)) for name, severity, ok, metric in BUILD_REPORT],
                      "check_name STRING, severity STRING, ok BOOLEAN, metric STRING") \
    .write.mode("overwrite").saveAsTable(f"{STG}._build_report")

In [0]:
FROM_SILVER = "source_table LIKE 'silver.%'"

# Every kind of row this notebook adds: table -> (id column, event date, how to recognise an added row).
BEYOND_ROWS = {
    "visit_occurrence": ("visit_occurrence_id", "visit_start_date",
                         "(visit_source_value = 'SILVER-NNU' OR visit_source_value LIKE 'ehr:%')"),
    "visit_detail": ("visit_detail_id", "visit_detail_start_date", FROM_SILVER),
    "condition_occurrence": ("condition_occurrence_id", "condition_start_date", FROM_SILVER),
    "procedure_occurrence": ("procedure_occurrence_id", "procedure_date", FROM_SILVER),
    "measurement": ("measurement_id", "measurement_date", FROM_SILVER),
    "observation": ("observation_id", "observation_date", FROM_SILVER),
    "drug_exposure": ("drug_exposure_id", "drug_exposure_start_date", FROM_SILVER),
    "device_exposure": ("device_exposure_id", "device_exposure_start_date", FROM_SILVER),
    "death": ("person_id", "death_date", f"cause_source_value IN ('{NNU_DEATH_SOURCE}', '{EHR_DEATH_SOURCE}')"),
}
# The added fact tables (rows that can carry a visit link).
BEYOND_FACT_TABLES = [t for t in BEYOND_ROWS if t not in ("visit_occurrence", "visit_detail", "death")]


def snapshot_added_rows():
    """Record which added rows exist before cleaning (STG._beyond_snapshot) and the release date (STG._clean_state);
    the reconciliation check (section 11) compares the published rows with them."""
    assert not spark.catalog.tableExists(f"{STG}._clean_state"), "this working schema has already been cleaned"
    seeded_release = spark.table(f"{STG}._seed_state").first().source_release
    assert str(seeded_release) <= RELEASE_DATE, \
        f"release date {RELEASE_DATE} is before the seeded MSDS release {seeded_release}"
    spark.sql(f"CREATE TABLE {STG}._beyond_snapshot AS " + " UNION ALL ".join(
        f"SELECT '{table}' AS t, {key} AS id FROM {STG}.{table} WHERE site_id = '{SITE}' AND {is_added}"
        for table, (key, _, is_added) in BEYOND_ROWS.items()))
    spark.sql(f"""CREATE TABLE {STG}._clean_state AS SELECT CAST(NULL AS BIGINT) AS all_facts_version,
      DATE'{RELEASE_DATE}' AS release_date, current_timestamp() AS cleaned_at""")


def drop_added_rows_outside_window():
    """Decide (STG._beyond_drop), record and delete the added rows that cannot be published."""
    people = f"""SELECT person_id, min(person_role) AS person_role, min(CAST(birth_datetime AS DATE)) AS bd
                 FROM {STG}.person WHERE site_id = '{SITE}' GROUP BY person_id"""
    deaths = f"SELECT person_id, min(death_date) AS dd FROM {STG}.death WHERE site_id = '{SITE}' GROUP BY person_id"
    decisions = []
    for table, (key, date_column, is_added) in BEYOND_ROWS.items():
        after_death = "" if table == "death" else f"WHEN x.{date_column} > date_add(dd.dd, 30) THEN 'after_death'"
        decisions.append(f"""
      SELECT '{table}' AS t, id, reason FROM (SELECT x.{key} AS id, CASE
          WHEN x.{date_column} < DATE'1900-01-01' THEN 'before_1900'
          WHEN x.{date_column} > DATE'{RELEASE_DATE}' THEN 'after_release'
          WHEN x.{date_column} < CASE WHEN p.person_role = 'infant' THEN date_sub(p.bd, 1) ELSE p.bd END THEN 'before_birth'
          {after_death} END AS reason
        FROM {STG}.{table} x
        LEFT JOIN ({people}) p ON p.person_id = x.person_id
        LEFT JOIN ({deaths}) dd ON dd.person_id = x.person_id
        WHERE x.site_id = '{SITE}' AND {is_added}) WHERE reason IS NOT NULL""")
    spark.sql(f"CREATE OR REPLACE TABLE {STG}._beyond_drop AS " + " UNION ALL ".join(decisions))

    # A visit detail must have its visit, so an added detail whose visit is dropped goes too.
    spark.sql(f"""INSERT INTO {STG}._beyond_drop
      SELECT 'visit_detail', vd.visit_detail_id, 'parent_visit_dropped' FROM {STG}.visit_detail vd
      WHERE vd.site_id = '{SITE}' AND vd.{FROM_SILVER}
        AND vd.visit_occurrence_id IN (SELECT id FROM {STG}._beyond_drop WHERE t = 'visit_occurrence')
        AND vd.visit_detail_id NOT IN (SELECT id FROM {STG}._beyond_drop WHERE t = 'visit_detail')""")

    record_dropped(f"SELECT concat('BEYOND:', t), CAST(id AS STRING), reason FROM {STG}._beyond_drop")
    for table, (key, _, is_added) in BEYOND_ROWS.items():
        spark.sql(f"""DELETE FROM {STG}.{table} WHERE site_id = '{SITE}' AND {is_added}
          AND {key} IN (SELECT id FROM {STG}._beyond_drop WHERE t = '{table}')""")
    spark.sql(f"""DELETE FROM {STG}._ehr_provenance p WHERE p.site_id = '{SITE}' AND EXISTS (
      SELECT 1 FROM {STG}._beyond_drop d WHERE d.t = p.cdm_table AND d.id = p.cdm_id)""")


def null_dangling_visit_links():
    """On surviving added facts, set visit_occurrence_id / visit_detail_id to NULL where that visit or detail is
    gone, recording each as FK:<table>.<column>."""
    for table in BEYOND_FACT_TABLES:
        key = BEYOND_ROWS[table][0]
        for column, referenced_table in [("visit_occurrence_id", "visit_occurrence"), ("visit_detail_id", "visit_detail")]:
            dangling = f"""{STG}.{table} x WHERE x.site_id = '{SITE}' AND x.{FROM_SILVER} AND x.{column} IS NOT NULL
              AND NOT EXISTS (SELECT 1 FROM {STG}.{referenced_table} r WHERE r.{column} = x.{column})"""
            record_dropped(f"SELECT 'FK:{table}.{column}', CAST(x.{key} AS STRING), 'nulled_dangling_fk' FROM {dangling}")
            spark.sql(f"""MERGE INTO {STG}.{table} f USING (SELECT x.{key} AS id FROM {dangling}) d ON f.{key} = d.id
              WHEN MATCHED THEN UPDATE SET {column} = NULL""")


snapshot_added_rows()
drop_added_rows_outside_window()
null_dangling_visit_links()
show(f"""SELECT rule_id, reason, count(*) n FROM {STG}._dropped
         WHERE rule_id LIKE 'BEYOND:%' OR rule_id LIKE 'FK:%' GROUP BY 1, 2 ORDER BY 1, 2""")

## 10. Assemble the OMOP tables

Every publish table is copied from `STG` to `PUB`. Then the tables that summarise the others are rebuilt from
the clean `PUB`: observation periods (now also covering drugs and devices), the pregnancy episode links
(`episode_event`, now also linking drug and device rows), the CDM source row and the bundled vocabulary.
Observation periods, the CDM source row and the vocabulary use the same code as `10_preborn_msds`.

In [0]:
def copy_staged_tables():
    """PUB.<table> = this site's STG.<table> rows, for every table this notebook publishes."""
    for table in BEYOND_PUBLISH_TABLES:
        columns = ", ".join(f"`{c}`" for c in table_columns(PUB, table))
        spark.sql(f"INSERT INTO {PUB}.{table} ({columns}) SELECT {columns} FROM {STG}.{table} WHERE site_id = '{SITE}'")
        print(table, spark.table(f"{PUB}.{table}").count())


copy_staged_tables()

In [0]:
# The dated columns that set a person's observation period: (table, date column).
OBSERVATION_PERIOD_DATES = [
    ("condition_occurrence", "condition_start_date"), ("measurement", "measurement_date"),
    ("observation", "observation_date"), ("procedure_occurrence", "procedure_date"),
    ("visit_occurrence", "visit_start_date"), ("visit_occurrence", "visit_end_date"),
    ("episode", "episode_start_date"), ("episode", "episode_end_date"),
    ("drug_exposure", "drug_exposure_start_date"), ("device_exposure", "device_exposure_start_date"),
]


def build_observation_periods(schema):
    """schema.observation_period for this site, from the published rows in schema (OBSERVATION_PERIOD_DATES)."""
    dated_rows = " UNION ALL ".join(
        f"SELECT person_id, {date_column} AS d FROM {schema}.{table} WHERE site_id = '{SITE}'"
        for table, date_column in OBSERVATION_PERIOD_DATES)
    spark.sql(f"DELETE FROM {schema}.observation_period WHERE site_id = '{SITE}'")
    spark.sql(f"""
    INSERT INTO {schema}.observation_period (observation_period_id, person_id,
      observation_period_start_date, observation_period_end_date, period_type_concept_id, site_id)
    WITH ev AS ({dated_rows}),
    span AS (
      SELECT person_id, min(d) AS first_date, max(d) AS last_date FROM ev WHERE d IS NOT NULL GROUP BY person_id
    ),
    pregnancies AS (
      SELECT person_id, min(pregnancy_start_date) AS ps,
             max(coalesce(pregnancy_end_date, pregnancy_start_date)) AS pe
      FROM {schema}.pregnancy WHERE site_id = '{SITE}' GROUP BY person_id
    )
    SELECT {mint_id('obs-period', 'p.person_id')}, p.person_id,
      coalesce(s.first_date, pf.ps, CAST(p.birth_datetime AS DATE)),
      coalesce(s.last_date, pf.pe, CAST(p.birth_datetime AS DATE)),
      32817, '{SITE}'
    FROM {schema}.person p
    LEFT JOIN span s ON s.person_id = p.person_id
    LEFT JOIN pregnancies pf ON pf.person_id = p.person_id
    WHERE p.site_id = '{SITE}'
    """)
    periods, people = spark.sql(f"""SELECT
        (SELECT count(*) FROM {schema}.observation_period WHERE site_id = '{SITE}'),
        (SELECT count(*) FROM {schema}.person WHERE site_id = '{SITE}')""").first()
    assert periods == people, f"{periods} observation periods for {people} people"
    print("observation periods:", periods)


def write_cdm_source(schema):
    """Replace schema.cdm_source with the PREBORN row for RELEASE_DATE."""
    spark.sql(f"""
      INSERT INTO {schema}.cdm_source BY NAME
      REPLACE WHERE true
      SELECT
        'PREBORN multi-site maternity OMOP subset' AS cdm_source_name,
        'PREBORN' AS cdm_source_abbreviation,
        'PREBORN participating NHS organisations' AS cdm_holder,
        'Multi-site maternity-focused OMOP CDM v5.4 subset: MSDS common core with optional site-specific enrichment'
          AS source_description,
        CAST(NULL AS STRING) AS source_documentation_reference,
        CAST(NULL AS STRING) AS cdm_etl_reference,
        DATE'{RELEASE_DATE}' AS source_release_date, DATE'{RELEASE_DATE}' AS cdm_release_date,
        'v5.4' AS cdm_version, 756265 AS cdm_version_concept_id,
        (SELECT vocabulary_version FROM {VOCAB}.vocabulary WHERE vocabulary_id = 'None') AS vocabulary_version
    """)


# The CDM field concept of each table's id column, for episode_event.
EPISODE_EVENT_LINKS = [
    ("condition_occurrence", "condition_occurrence_id", 1147127),
    ("procedure_occurrence", "procedure_occurrence_id", 1147082),
    ("measurement", "measurement_id", 1147138),
    ("observation", "observation_id", 1147165),
    ("visit_occurrence", "visit_occurrence_id", 1147070),
    ("visit_detail", "visit_detail_id", 1147624),
    ("drug_exposure", "drug_exposure_id", 1147094),
    ("device_exposure", "device_exposure_id", 1147115),
]


def rebuild_episode_events():
    """Replace PUB.episode_event for this site: every row with a pregnancy_id, linked to its pregnancy's episode."""
    spark.sql(f"DELETE FROM {PUB}.episode_event WHERE site_id = '{SITE}'")
    for table, id_column, field_concept in EPISODE_EVENT_LINKS:
        spark.sql(f"""
        INSERT INTO {PUB}.episode_event (episode_id, event_id, episode_event_field_concept_id, source_table, site_id)
        SELECT e.episode_id, f.{id_column}, {field_concept}, '{table}', '{SITE}'
        FROM {PUB}.{table} f
        JOIN {PUB}.episode e ON e.pregnancy_id = f.pregnancy_id AND e.site_id = f.site_id
        WHERE f.site_id = '{SITE}' AND f.pregnancy_id IS NOT NULL
        """)


build_observation_periods(PUB)
rebuild_episode_events()
write_cdm_source(PUB)

In [0]:
# Every concept-carrying column, per table (as in 10_preborn_msds, plus the extension's tables).
CONCEPT_COLUMNS = {
    "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
               "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
    "observation_period": ["period_type_concept_id"],
    "condition_occurrence": ["condition_concept_id", "condition_type_concept_id",
                             "condition_status_concept_id", "condition_source_concept_id"],
    "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id",
                             "modifier_concept_id", "procedure_source_concept_id"],
    "measurement": ["measurement_concept_id", "measurement_type_concept_id", "operator_concept_id",
                    "value_as_concept_id", "unit_concept_id", "measurement_source_concept_id",
                    "unit_source_concept_id", "meas_event_field_concept_id"],
    "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                    "qualifier_concept_id", "unit_concept_id", "observation_source_concept_id",
                    "obs_event_field_concept_id"],
    "visit_occurrence": ["visit_concept_id", "visit_type_concept_id", "visit_source_concept_id",
                         "admitted_from_concept_id", "discharged_to_concept_id"],
    "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id",
                     "visit_detail_source_concept_id", "admitted_from_concept_id",
                     "discharged_to_concept_id"],
    "death": ["death_type_concept_id", "cause_concept_id", "cause_source_concept_id"],
    "location": ["country_concept_id"],
    "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id",
                "episode_source_concept_id"],
    "episode_event": ["episode_event_field_concept_id"],
    "fact_relationship": ["domain_concept_id_1", "domain_concept_id_2", "relationship_concept_id"],
    "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                  "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                  "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
    "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
    "cdm_source": ["cdm_version_concept_id"],
    "_msds_concept_map": ["concept_id"],
}
CONCEPT_COLUMNS.update({
    "drug_exposure": ["drug_concept_id", "drug_type_concept_id", "route_concept_id", "drug_source_concept_id"],
    "device_exposure": ["device_concept_id", "device_type_concept_id", "unit_concept_id",
                        "device_source_concept_id", "unit_source_concept_id"],
    "_concept_set": ["concept_id"],
})


def missing_reference_checks(schema):
    """(name, SQL counting dangling references) for every vocabulary link inside schema's bundle."""
    def missing(child, column, parent, parent_column, positive_only=False):
        only_positive = f" AND x.{column} > 0" if positive_only else ""
        return (f"(SELECT count(*) FROM {schema}.{child} x LEFT JOIN {schema}.{parent} y "
                f"ON y.{parent_column} = x.{column} WHERE y.{parent_column} IS NULL{only_positive})")
    return [
        ("concept_relationship.concept_id_1 in concept", missing("concept_relationship", "concept_id_1", "concept", "concept_id")),
        ("concept_relationship.concept_id_2 in concept", missing("concept_relationship", "concept_id_2", "concept", "concept_id")),
        ("concept_ancestor.ancestor in concept", missing("concept_ancestor", "ancestor_concept_id", "concept", "concept_id")),
        ("concept_ancestor.descendant in concept", missing("concept_ancestor", "descendant_concept_id", "concept", "concept_id")),
        ("vocabulary.vocabulary_concept_id in concept", missing("vocabulary", "vocabulary_concept_id", "concept", "concept_id", True)),
        ("domain.domain_concept_id in concept", missing("domain", "domain_concept_id", "concept", "concept_id", True)),
        ("concept_class.concept_class_concept_id in concept", missing("concept_class", "concept_class_concept_id", "concept", "concept_id", True)),
        ("relationship.relationship_concept_id in concept", missing("relationship", "relationship_concept_id", "concept", "concept_id", True)),
        ("concept_synonym.concept_id in concept", missing("concept_synonym", "concept_id", "concept", "concept_id")),
        ("concept_synonym.language_concept_id in concept", missing("concept_synonym", "language_concept_id", "concept", "concept_id", True)),
        ("cdm_source.cdm_version_concept_id in concept", missing("cdm_source", "cdm_version_concept_id", "concept", "concept_id", True)),
        ("concept.domain_id in domain", missing("concept", "domain_id", "domain", "domain_id")),
        ("concept.vocabulary_id in vocabulary", missing("concept", "vocabulary_id", "vocabulary", "vocabulary_id")),
        ("concept.concept_class_id in concept_class", missing("concept", "concept_class_id", "concept_class", "concept_class_id")),
    ]


def build_bundled_vocabulary(schema):
    """Rebuild schema's vocabulary tables from VOCAB for every concept the schema uses (see above)."""
    # 1. The concepts the schema uses.
    used = " UNION ALL ".join(f"SELECT CAST({column} AS BIGINT) AS cid FROM {schema}.{table}"
                              for table, columns in CONCEPT_COLUMNS.items() for column in columns)
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_used AS
      SELECT DISTINCT cid FROM ({used}) WHERE cid IS NOT NULL AND cid > 0""")

    # 2. Plus their ancestors and the concepts the metadata tables point at.
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_ids AS
      SELECT cid FROM {schema}._vocab_used
      UNION
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) FROM {VOCAB}.concept_ancestor ca
        JOIN {schema}._vocab_used u ON ca.descendant_concept_id = u.cid
      UNION
      SELECT CAST(vocabulary_concept_id AS BIGINT) FROM {VOCAB}.vocabulary WHERE vocabulary_concept_id > 0
      UNION
      SELECT CAST(domain_concept_id AS BIGINT) FROM {VOCAB}.domain WHERE domain_concept_id > 0
      UNION
      SELECT CAST(concept_class_concept_id AS BIGINT) FROM {VOCAB}.concept_class WHERE concept_class_concept_id > 0
      UNION
      SELECT CAST(relationship_concept_id AS BIGINT) FROM {VOCAB}.relationship WHERE relationship_concept_id > 0""")

    # 3. Plus the language concepts of the included concepts' synonyms, repeated until nothing new is added.
    for _ in range(5):
        before = spark.table(f"{schema}._vocab_ids").count()
        spark.sql(f"""CREATE OR REPLACE TABLE {schema}._vocab_ids_next AS
          SELECT cid FROM {schema}._vocab_ids
          UNION
          SELECT CAST(cs.language_concept_id AS BIGINT)
          FROM {VOCAB}.concept_synonym cs JOIN {schema}._vocab_ids v ON cs.concept_id = v.cid
          WHERE cs.language_concept_id IS NOT NULL AND cs.language_concept_id > 0""")
        spark.sql(f"CREATE OR REPLACE TABLE {schema}._vocab_ids AS SELECT cid FROM {schema}._vocab_ids_next")
        if spark.table(f"{schema}._vocab_ids").count() == before:
            break
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_ids_next")

    # 4. Write the vocabulary tables.
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept AS
      SELECT CAST(c.concept_id AS BIGINT) AS concept_id, c.concept_name, c.domain_id, c.vocabulary_id,
             c.concept_class_id, c.standard_concept, c.concept_code,
             c.valid_start_date, c.valid_end_date, c.invalid_reason
      FROM {VOCAB}.concept c JOIN {schema}._vocab_ids v ON c.concept_id = v.cid""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_relationship AS
      SELECT CAST(cr.concept_id_1 AS BIGINT) AS concept_id_1, CAST(cr.concept_id_2 AS BIGINT) AS concept_id_2,
             cr.relationship_id, cr.valid_start_date, cr.valid_end_date, cr.invalid_reason
      FROM {VOCAB}.concept_relationship cr
      WHERE cr.concept_id_1 IN (SELECT concept_id FROM {schema}.concept)
        AND cr.concept_id_2 IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_ancestor AS
      SELECT CAST(ca.ancestor_concept_id AS BIGINT) AS ancestor_concept_id,
             CAST(ca.descendant_concept_id AS BIGINT) AS descendant_concept_id,
             CAST(ca.min_levels_of_separation AS BIGINT) AS min_levels_of_separation,
             CAST(ca.max_levels_of_separation AS BIGINT) AS max_levels_of_separation
      FROM {VOCAB}.concept_ancestor ca
      WHERE ca.descendant_concept_id IN (SELECT cid FROM {schema}._vocab_used)
        AND ca.ancestor_concept_id IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_synonym AS
      SELECT CAST(cs.concept_id AS BIGINT) AS concept_id, cs.concept_synonym_name,
             CAST(cs.language_concept_id AS BIGINT) AS language_concept_id
      FROM {VOCAB}.concept_synonym cs
      WHERE cs.concept_id IN (SELECT concept_id FROM {schema}.concept)""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.vocabulary AS
      SELECT vocabulary_id, vocabulary_name, vocabulary_reference, vocabulary_version,
             CAST(vocabulary_concept_id AS BIGINT) AS vocabulary_concept_id
      FROM {VOCAB}.vocabulary""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.domain AS
      SELECT domain_id, domain_name, CAST(domain_concept_id AS BIGINT) AS domain_concept_id FROM {VOCAB}.domain""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.concept_class AS
      SELECT concept_class_id, concept_class_name, CAST(concept_class_concept_id AS BIGINT) AS concept_class_concept_id
      FROM {VOCAB}.concept_class""")
    spark.sql(f"""CREATE OR REPLACE TABLE {schema}.relationship AS
      SELECT relationship_id, relationship_name, is_hierarchical, defines_ancestry,
             reverse_relationship_id, CAST(relationship_concept_id AS BIGINT) AS relationship_concept_id
      FROM {VOCAB}.relationship""")

    # 5. Check that nothing dangles.
    used_count = spark.table(f"{schema}._vocab_used").count()
    concept_count = spark.table(f"{schema}.concept").count()
    print(f"bundled vocabulary for {schema}: {used_count} concepts used, {concept_count} concepts bundled")
    dangling = 0
    for name, count_sql in missing_reference_checks(schema):
        n = scalar(f"SELECT {count_sql}")
        dangling += n
        print(f"  {'OK  ' if n == 0 else 'FAIL'} {name}: {n} dangling")
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_used")
    spark.sql(f"DROP TABLE IF EXISTS {schema}._vocab_ids")
    assert dangling == 0, f"the bundled vocabulary in {schema} has {dangling} dangling references"


build_bundled_vocabulary(PUB)

## 11. Quality checks

Read-only checks, run on the staged `PUB` schema here and on the target straight after publishing (section
12), where a failure rolls the publish back. Results are appended to `{target}._qa_results`, together with the
lanes' report rows. **hard** checks stop the run; **report** checks are logged.

| Group | Checks |
|---|---|
| structure | the same checks as `10_preborn_msds` (keys, references, required columns, observation periods, concepts, vocabulary links), also for drugs and devices |
| visits | visits and details end on or after their start; details sit within their visit |
| preservation | every MSDS row seeded in section 4 is in the schema unchanged, apart from the allowed NULL fills |
| person links | at least 90% of mothers and 65% of babies are linked to silver |
| reconciliation | every added row is published or dropped (never both, never neither), nothing unstaged is published, and nothing published is outside the time window |
| lineage | each added row has exactly one provenance row; each EHR fact's concept is in the concept set for the person's role; each EHR row is on a linked person |

In [0]:
QA_RESULTS_COLUMNS = """site_id STRING, run_tag STRING, source_release DATE, scope STRING, check_name STRING,
  severity STRING, ok BOOLEAN, metric STRING, checked_at TIMESTAMP"""

# Per table: (primary key, the references it holds, the columns that must be filled for this site).
TABLE_RULES = {
    "person": ("person_id", [], ["gender_concept_id", "year_of_birth"]),
    "observation_period": ("observation_period_id", ["person"], ["period_type_concept_id"]),
    "condition_occurrence": ("condition_occurrence_id", ["person", "pregnancy", "visit"],
                             ["condition_start_date", "condition_type_concept_id"]),
    "procedure_occurrence": ("procedure_occurrence_id", ["person", "pregnancy", "visit"],
                             ["procedure_date", "procedure_type_concept_id"]),
    "measurement": ("measurement_id", ["person", "pregnancy", "visit"],
                    ["measurement_date", "measurement_type_concept_id"]),
    "observation": ("observation_id", ["person", "pregnancy", "visit"],
                    ["observation_date", "observation_type_concept_id"]),
    "visit_occurrence": ("visit_occurrence_id", ["person", "pregnancy"],
                         ["person_id", "visit_concept_id", "visit_start_date", "visit_end_date",
                          "visit_type_concept_id"]),
    "visit_detail": ("visit_detail_id", ["person", "pregnancy", "visit"],
                     ["visit_occurrence_id", "visit_detail_start_date", "visit_detail_type_concept_id"]),
    "episode": ("episode_id", ["person", "pregnancy"], ["episode_type_concept_id"]),
    "death": ("person_id", ["person"], ["death_date", "death_type_concept_id"]),
}
TABLE_RULES.update({
    "drug_exposure": ("drug_exposure_id", ["person", "pregnancy", "visit"],
                      ["drug_exposure_start_date", "drug_exposure_end_date", "drug_type_concept_id"]),
    "device_exposure": ("device_exposure_id", ["person", "pregnancy", "visit"],
                        ["device_exposure_start_date", "device_type_concept_id"]),
})
# A reference: (column holding it, the table it points at, that table's key).
REFERENCES = {
    "person": ("person_id", "person", "person_id"),
    "pregnancy": ("pregnancy_id", "pregnancy", "pregnancy_id"),
    "visit": ("visit_occurrence_id", "visit_occurrence", "visit_occurrence_id"),
}
# The concept columns whose values must exist in the bundled concept table.
QA_CONCEPT_COLUMNS = {
    "person": ["gender_concept_id", "race_concept_id", "ethnicity_concept_id",
               "gender_source_concept_id", "race_source_concept_id", "ethnicity_source_concept_id"],
    "observation_period": ["period_type_concept_id"],
    "condition_occurrence": ["condition_concept_id", "condition_type_concept_id", "condition_source_concept_id"],
    "procedure_occurrence": ["procedure_concept_id", "procedure_type_concept_id", "procedure_source_concept_id"],
    "measurement": ["measurement_concept_id", "measurement_type_concept_id", "value_as_concept_id",
                    "unit_concept_id", "measurement_source_concept_id"],
    "observation": ["observation_concept_id", "observation_type_concept_id", "value_as_concept_id",
                    "unit_concept_id", "observation_source_concept_id"],
    "visit_occurrence": ["visit_concept_id", "visit_type_concept_id"],
    "visit_detail": ["visit_detail_concept_id", "visit_detail_type_concept_id"],
    "episode": ["episode_concept_id", "episode_object_concept_id", "episode_type_concept_id"],
    "death": ["death_type_concept_id", "cause_concept_id"],
    "location": ["country_concept_id"],
    "pregnancy": ["pregnancy_outcome_concept_id", "pregnancy_mode_delivery_concept_id",
                  "pregnancy_single_concept_id", "pregnancy_marital_status_concept_id",
                  "prev_pregnancy_parity_concept_id", "pregnancy_folic_concept_id"],
    "infant": ["birth_outcome_concept_id", "birth_con_malformation_concept_id"],
    "_msds_concept_map": ["concept_id"],
}
QA_CONCEPT_COLUMNS.update({
    "drug_exposure": ["drug_concept_id", "drug_type_concept_id", "route_concept_id", "drug_source_concept_id"],
    "device_exposure": ["device_concept_id", "device_type_concept_id", "device_source_concept_id"],
})

In [0]:
def qa_structure(schema):
    """Keys, references, required columns, observation-period coverage, concepts and vocabulary links.
    Returns a list of (check name, severity, passed, metric). One query per table keeps this quick."""
    results = []
    for table, (key, references, required) in TABLE_RULES.items():
        joins = []
        measures = ["count(*) AS n", f"count(DISTINCT f.{key}) AS d"]
        for ref in references:
            column, ref_table, ref_key = REFERENCES[ref]
            joins.append(f"LEFT JOIN (SELECT DISTINCT {ref_key} FROM {schema}.{ref_table}) j_{ref} "
                         f"ON j_{ref}.{ref_key} = f.{column}")
            measures.append(f"count_if(f.{column} IS NOT NULL AND j_{ref}.{ref_key} IS NULL) AS o_{ref}")
        for column in required:
            measures.append(f"count_if(f.site_id = '{SITE}' AND f.{column} IS NULL) AS nn_{column}")
        if table == "person":
            joins.append(f"LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.observation_period) op ON op.person_id = f.person_id")
            joins.append(f"LEFT JOIN (SELECT DISTINCT location_id FROM {schema}.location) lo ON lo.location_id = f.location_id")
            measures.append(f"count_if(f.site_id = '{SITE}' AND op.person_id IS NULL) AS uncovered")
            measures.append("count_if(f.location_id IS NOT NULL AND lo.location_id IS NULL) AS o_location")
        row = spark.sql(f"SELECT {', '.join(measures)} FROM {schema}.{table} f {' '.join(joins)}").first().asDict()

        results.append((f"{table}.{key} unique (global)", "hard", row["n"] == row["d"], f"{row['n']}/{row['d']}"))
        for ref in references:
            ref_table = REFERENCES[ref][1]
            results.append((f"{table} -> {ref_table}", "hard", row[f"o_{ref}"] == 0, f"{row[f'o_{ref}']} orphans"))
        if table == "person":
            results.append(("person -> location", "hard", row["o_location"] == 0, f"{row['o_location']} orphans"))
        for column in required:
            results.append((f"{table}.{column} not-null", "hard", row[f"nn_{column}"] == 0, f"{row[f'nn_{column}']} null"))
        if table == "person":
            results.append(("observation_period covers every person", "hard", row["uncovered"] == 0,
                            f"{row['uncovered']} uncovered"))

    row = spark.sql(f"""SELECT
      (SELECT count(*) FROM {schema}.episode_event ee
         LEFT JOIN (SELECT DISTINCT episode_id FROM {schema}.episode) e ON e.episode_id = ee.episode_id
         WHERE e.episode_id IS NULL) AS ee_orphan,
      (SELECT count(*) FROM {schema}.fact_relationship fr
         LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.person) p1 ON p1.person_id = fr.fact_id_1
         LEFT JOIN (SELECT DISTINCT person_id FROM {schema}.person) p2 ON p2.person_id = fr.fact_id_2
         WHERE p1.person_id IS NULL OR p2.person_id IS NULL) AS fr_orphan,
      (SELECT count_if(site_id = '{SITE}') - count(DISTINCT CASE WHEN site_id = '{SITE}' THEN location_id END)
         FROM {schema}.location) AS loc_dup_site,
      (SELECT count(*) - count(DISTINCT location_id) FROM {schema}.location) AS loc_dup_global,
      (SELECT count(*) FROM {schema}._msds_concept_map WHERE site_id = '{SITE}') AS concept_map_rows""").first()
    results += [
        ("episode_event -> episode", "hard", row.ee_orphan == 0, f"{row.ee_orphan} orphans"),
        ("fact_relationship endpoints -> person", "hard", row.fr_orphan == 0, f"{row.fr_orphan} orphans"),
        ("location.location_id unique (site)", "hard", row.loc_dup_site == 0, f"{row.loc_dup_site} duplicates"),
        # LSOA ids are shared codes, so two sites can rightly publish the same LSOA row
        ("location.location_id unique (global)", "report", row.loc_dup_global == 0, f"{row.loc_dup_global} duplicates"),
        ("_msds_concept_map populated", "report", row.concept_map_rows > 0, f"{row.concept_map_rows} rows"),
    ]

    used = " UNION ALL ".join(f"SELECT CAST({column} AS BIGINT) AS cid FROM {schema}.{table} WHERE site_id = '{SITE}'"
                              for table, columns in QA_CONCEPT_COLUMNS.items() for column in columns)
    missing = scalar(f"""SELECT count(*) FROM (SELECT DISTINCT cid FROM ({used}) WHERE cid > 0) u
                         LEFT JOIN {schema}.concept c ON c.concept_id = u.cid WHERE c.concept_id IS NULL""")
    results.append(("all used non-zero concept ids exist in concept", "hard", missing == 0, f"{missing} missing"))

    checks = missing_reference_checks(schema)
    counts = spark.sql("SELECT " + ", ".join(f"{sql} AS c{i}" for i, (_, sql) in enumerate(checks))).first()
    for i, (name, _) in enumerate(checks):
        results.append((name, "hard", counts[i] == 0, f"{counts[i]} dangling"))
    return results


def qa_visits(schema):
    """Visit and detail dates in order; details inside their visit. (Whether MSDS visits came from held contacts
    was checked by 10_preborn_msds; the preservation check proves they are unchanged.)"""
    row = spark.sql(f"""SELECT
      (SELECT count_if(visit_end_date < visit_start_date) FROM {schema}.visit_occurrence
         WHERE site_id = '{SITE}') AS vo_neg,
      (SELECT count_if(visit_detail_end_date < visit_detail_start_date) FROM {schema}.visit_detail
         WHERE site_id = '{SITE}') AS vd_neg,
      (SELECT count(*) FROM {schema}.visit_detail vd JOIN {schema}.visit_occurrence vo
         ON vo.visit_occurrence_id = vd.visit_occurrence_id
         WHERE vd.site_id = '{SITE}' AND (vd.visit_detail_start_date < date_sub(vo.visit_start_date, 1)
           OR coalesce(vd.visit_detail_end_date, vd.visit_detail_start_date) > date_add(vo.visit_end_date, 1))
      ) AS vd_outside""").first()
    return [
        ("visit_occurrence end >= start", "hard", row.vo_neg == 0, f"{row.vo_neg} rows"),
        ("visit_detail end >= start", "hard", row.vd_neg == 0, f"{row.vd_neg} rows"),
        ("visit_detail within its visit (+/-1 day)", "report", row.vd_outside == 0, f"{row.vd_outside} rows"),
    ]


def row_hash_sql(columns, alias):
    """SQL hash of a row's values in the given columns (NULL-safe), for comparing rows."""
    parts = ", ".join(f"coalesce(cast({alias}.`{c}` AS STRING), chr(0))" for c in columns)
    return f"xxhash64(concat_ws(chr(31), {parts}))"


def qa_preservation(schema):
    """Every MSDS row seeded in section 4 is in schema unchanged, except NULL -> value fills in FILL_COLUMNS."""
    seed = spark.table(f"{STG}._seed_state").first()
    versions = json.loads(seed.versions)
    results = [("seed pinned to published MSDS run", "report", True,
                f"{seed.msds_schema} {seed.msds_run_tag} versions {seed.versions}")]
    for table, key in SEED_KEYS.items():
        msds_rows = f"(SELECT * FROM {seed.msds_schema}.{table} VERSION AS OF {versions[table]} WHERE site_id = '{seed.site_id}')"
        columns = table_columns(seed.msds_schema, table)
        fills = FILL_COLUMNS.get(table, [])
        protected = [c for c in columns if c not in fills]
        if key is None:
            # No key: every MSDS row must appear at least as often in schema (multiset containment of row hashes).
            r = spark.sql(f"""
              WITH m AS (SELECT {row_hash_sql(columns, 'x')} h, count(*) n FROM {msds_rows} x GROUP BY 1),
                   b AS (SELECT {row_hash_sql(columns, 'x')} h, count(*) n FROM {schema}.{table} x
                         WHERE site_id = '{seed.site_id}' GROUP BY 1)
              SELECT coalesce(sum(m.n), 0) AS n, coalesce(sum(greatest(m.n - coalesce(b.n, 0), 0)), 0) AS bad
              FROM m LEFT JOIN b ON b.h = m.h""").first()
            results.append((f"preserved {table}", "hard", r.bad == 0, f"{r.bad} of {r.n} MSDS rows missing or changed"))
            continue
        fill_changed = " + ".join(f"count_if(m.`{c}` IS NOT NULL AND b.`{c}` IS DISTINCT FROM m.`{c}`)"
                                  for c in fills) or "0"
        r = spark.sql(f"""SELECT count(*) AS n, count(b.`{key}`) AS found,
            count_if(b.`{key}` IS NOT NULL AND {row_hash_sql(protected, 'b')} <> {row_hash_sql(protected, 'm')}) AS changed,
            {fill_changed} AS fill_bad
          FROM {msds_rows} m
          LEFT JOIN (SELECT * FROM {schema}.{table} WHERE site_id = '{seed.site_id}') b ON b.`{key}` = m.`{key}`""").first()
        ok = r.found == r.n and r.changed == 0 and r.fill_bad == 0
        results.append((f"preserved {table}", "hard", ok,
                        f"{r.n} MSDS rows: {r.n - r.found} missing, {r.changed} changed, {r.fill_bad} fills overwrote MSDS"))
    return results


def qa_bridge():
    """The share of mothers and babies linked to silver must reach BRIDGE_FLOORS."""
    rows = spark.sql(f"SELECT role, count(*) n, count(silver_person_id) linked FROM {STG}._bridge GROUP BY role").collect()
    by_method = {(r.role, r.method): r.n for r in spark.sql(
        f"SELECT role, method, count(*) n FROM {STG}._bridge GROUP BY 1, 2").collect()}
    results = []
    for r in rows:
        rate = r.linked / r.n if r.n else 0.0
        methods = ", ".join(f"{method} {n}" for (role, method), n in sorted(by_method.items()) if role == r.role)
        results.append((f"bridge {r.role} link rate >= {BRIDGE_FLOORS[r.role]:.0%}", "hard",
                        rate >= BRIDGE_FLOORS[r.role], f"{r.linked}/{r.n} = {rate:.1%} ({methods})"))
    for role in sorted(set(BRIDGE_FLOORS) - {r.role for r in rows}):
        results.append((f"bridge {role} link rate >= {BRIDGE_FLOORS[role]:.0%}", "hard", False, "no rows"))
    return results


def qa_reconcile(schema):
    """Every added row staged before cleaning is published XOR dropped; none is published that was not staged;
    no published added row falls outside the release window."""
    release = spark.table(f"{STG}._clean_state").first().release_date
    published = " UNION ALL ".join(
        f"SELECT '{table}' AS t, {key} AS id, {date_column} AS d FROM {schema}.{table} WHERE site_id = '{SITE}' AND {is_added}"
        for table, (key, date_column, is_added) in BEYOND_ROWS.items())
    rows = spark.sql(f"""WITH p AS ({published}),
      drp AS (SELECT DISTINCT substring(rule_id, 8) AS t, CAST(fact_natural_key AS BIGINT) AS id
              FROM {STG}._dropped WHERE rule_id LIKE 'BEYOND:%'),
      sn AS (SELECT DISTINCT t, id FROM {STG}._beyond_snapshot),
      a AS (SELECT coalesce(sn.t, p.t) AS t, sn.id IS NOT NULL AS staged, p.id IS NOT NULL AS published,
                   d.id IS NOT NULL AS dropped, p.d AS d
            FROM sn FULL JOIN p ON p.t = sn.t AND p.id = sn.id
            LEFT JOIN drp d ON d.t = coalesce(sn.t, p.t) AND d.id = coalesce(sn.id, p.id))
      SELECT t, count_if(staged) n, count_if(published) pub, count_if(dropped) drp,
             count_if(staged AND NOT published AND NOT dropped) gap, count_if(published AND dropped) both_,
             count_if(published AND NOT staged) extra,
             count_if(published AND (d < DATE'1900-01-01' OR d > DATE'{release}')) outside
      FROM a GROUP BY t ORDER BY t""").collect()
    results = []
    for x in rows:
        ok = x.gap == 0 and x.both_ == 0 and x.extra == 0 and x.outside == 0
        results.append((f"reconcile beyond {x.t}", "hard", ok,
                        f"n {x.n} = published {x.pub} + dropped {x.drp}; gap {x.gap}, both {x.both_}, extra {x.extra}, "
                        f"outside window {x.outside}"))
    seen = {x.t for x in rows}
    for table in BEYOND_ROWS:
        if table not in seen:
            results.append((f"reconcile beyond {table}", "report", True, "no beyond rows"))
    for x in spark.sql(f"""SELECT rule_id, reason, count(*) n FROM {STG}._dropped
                           WHERE rule_id LIKE 'BEYOND:%' OR rule_id LIKE 'FK:%' GROUP BY 1, 2 ORDER BY 1, 2""").collect():
        results.append((f"dropped: {x.rule_id} {x.reason}", "report", True, f"{x.n} rows"))
    return results


def qa_lineage(schema):
    """Provenance: one row per added row, each pointing at a real row. EHR concepts inside the concept set for the
    person's role. Every EHR-lane row on a person linked to silver."""
    with_provenance = {"visit_occurrence": "visit_source_value = 'SILVER-NNU'",
                       "death": f"cause_source_value = '{NNU_DEATH_SOURCE}'"}
    for table in BEYOND_FACT_TABLES + ["visit_detail"]:
        with_provenance[table] = FROM_SILVER
    added = " UNION ALL ".join(
        f"SELECT '{table}' AS t, {BEYOND_ROWS[table][0]} AS id FROM {schema}.{table} WHERE site_id = '{SITE}' AND {is_added}"
        for table, is_added in with_provenance.items())
    r = spark.sql(f"""WITH f AS ({added}),
      p AS (SELECT cdm_table AS t, cdm_id AS id, count(*) n FROM {schema}._ehr_provenance
            WHERE site_id = '{SITE}' GROUP BY 1, 2)
      SELECT count_if(p.id IS NULL) AS no_prov, count_if(f.id IS NULL) AS orphan_prov, count_if(p.n > 1) AS dup_prov
      FROM f FULL JOIN p ON p.t = f.t AND p.id = f.id""").first()
    results = [("every beyond fact has provenance", "hard", r.no_prov == 0, f"{r.no_prov} rows without"),
               ("every provenance row resolves to a published row", "hard", r.orphan_prov == 0, f"{r.orphan_prov} orphans"),
               ("provenance one row per fact", "hard", r.dup_prov == 0, f"{r.dup_prov} duplicated")]

    per_source = []
    for spec in EHR_SPECS:
        concept_column = next(column for column, expression in spec["cols"].items() if expression == spec["cid"])
        per_source.append(f"""SELECT '{spec['src']}' AS src, count(*) n, count_if(c.concept_id IS NULL) AS outside
          FROM {schema}.{spec['table']} x JOIN {STG}._bridge b ON b.person_id = x.person_id
          LEFT JOIN {schema}._concept_set c ON c.site_id = '{SITE}' AND c.concept_id = x.{concept_column} AND c.role = b.role
          WHERE x.site_id = '{SITE}' AND x.source_table = 'silver.{spec['src']}'""")
    for x in spark.sql(" UNION ALL ".join(per_source)).collect():
        results.append((f"ehr {x.src} concepts inside the concept set", "hard", x.outside == 0,
                        f"{x.outside} of {x.n} outside"))

    # EHR-lane rows come through the bridge (neonatal rows may come through the mother instead, see section 6).
    ehr_rows = {table: f"source_table IN ({EHR_SOURCES_IN})" for table in BEYOND_FACT_TABLES}
    ehr_rows["visit_occurrence"] = "visit_source_value LIKE 'ehr:%'"
    ehr_rows["death"] = f"cause_source_value = '{EHR_DEATH_SOURCE}'"
    unlinked = scalar("SELECT " + " + ".join(
        f"""(SELECT count(*) FROM {schema}.{table} x WHERE x.site_id = '{SITE}' AND x.{is_ehr} AND NOT EXISTS (
              SELECT 1 FROM {STG}._bridge b WHERE b.person_id = x.person_id AND b.silver_person_id IS NOT NULL))"""
        for table, is_ehr in ehr_rows.items()))
    results.append(("every EHR-lane row is on a bridged person", "hard", unlinked == 0, f"{unlinked} rows"))
    return results


def run_qa(schema, scope):
    """Run every check on schema, append the results and the lanes' report rows to TARGET._qa_results, print
    them, and raise if any hard check failed. scope is 'staged' (the PUB schema) or 'published' (the target)."""
    assert scope in ("staged", "published"), scope
    results = (qa_structure(schema) + qa_visits(schema) + qa_preservation(schema) + qa_bridge()
               + qa_reconcile(schema) + qa_lineage(schema)
               + [tuple(row) for row in spark.table(f"{STG}._build_report").collect()])

    release = spark.table(f"{STG}._clean_state").first().release_date
    now = datetime.now(timezone.utc).replace(tzinfo=None)
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {TARGET}")
    spark.sql(f"CREATE TABLE IF NOT EXISTS {TARGET}._qa_results ({QA_RESULTS_COLUMNS}) USING DELTA")
    rows = [(SITE_ID, RUN_TAG, release, scope, name, severity, bool(ok), str(metric), now)
            for name, severity, ok, metric in results]
    spark.createDataFrame(rows, QA_RESULTS_COLUMNS.replace("\n", " ")) \
        .write.mode("append").saveAsTable(f"{TARGET}._qa_results")

    for name, severity, ok, metric in results:
        status = "PASS" if ok else ("FAIL" if severity == "hard" else "WARN")
        print(f"{status}  {name}: {metric}")
    failures = [name for name, severity, ok, _ in results if severity == "hard" and not ok]
    if failures:
        raise AssertionError(f"PREBORN QA ({scope}, site {SITE_ID}): {len(failures)} hard failures: {failures}")
    print(f"PREBORN QA ({scope}, site {SITE_ID}): {len(results)} checks, 0 hard failures")


run_qa(PUB, "staged")

## 12. Publish

The same publish as `10_preborn_msds` (copied; the test suite checks the copies match): one writer at a time,
every table's version journalled first, everything restored if any step or the post-publish checks fail, and an
interrupted earlier publish recovered before anything else. This site's rows of every publish table are replaced;
the target's bundled vocabulary is rebuilt across all sites in it.

In [0]:
# Unity Catalog descriptions for the published tables and columns. "*" applies to a column name in any table;
# a per-table entry overrides it.
PREBORN_TABLE_COMMENTS = {
    "person": "OMOP CDM PERSON records for pregnant people and infants in PREBORN, with source-site and maternal/infant role lineage.",
    "observation_period": "OMOP CDM OBSERVATION_PERIOD records defining the dates during which each PREBORN person is represented by available clinical data.",
    "condition_occurrence": "OMOP CDM CONDITION_OCCURRENCE records generated from PREBORN maternity and neonatal diagnosis data.",
    "procedure_occurrence": "OMOP CDM PROCEDURE_OCCURRENCE records generated from PREBORN maternity and neonatal care activities.",
    "measurement": "OMOP CDM MEASUREMENT records for structured numeric or coded maternity and neonatal measurements.",
    "observation": "OMOP CDM OBSERVATION records for clinical findings and other non-measurement facts in PREBORN.",
    "visit_occurrence": "OMOP CDM VISIT_OCCURRENCE records representing maternity care contacts linked to people and pregnancies.",
    "visit_detail": "OMOP CDM VISIT_DETAIL records representing detailed neonatal or enrichment encounters within a visit.",
    "death": "OMOP CDM DEATH records for mothers and infants from MSDS, earliest recorded death per person.",
    "location": "OMOP CDM LOCATION records at LSOA 2011 granularity (England/Wales), derived from the mother's MSDS residence; no postcode or address is held.",
    "episode": "OMOP CDM EPISODE records representing pregnancy episodes and their clinical date bounds.",
    "episode_event": "OMOP CDM EPISODE_EVENT bridge linking pregnancy episodes to their constituent clinical events.",
    "fact_relationship": "OMOP CDM FACT_RELATIONSHIP rows linking related PREBORN facts, including pregnancy and infant relationships.",
    "cdm_source": "OMOP CDM_SOURCE metadata describing the PREBORN CDM instance, ETL release and vocabulary versions.",
    "pregnancy": "PREBORN maternity extension containing one row per pregnancy with gestation, outcome, delivery and prior-pregnancy attributes.",
    "infant": "PREBORN maternity extension containing one row per infant with birth outcome, weight, Apgar and congenital-malformation attributes.",
    "concept": "Bundled OMOP CONCEPT vocabulary subset containing concepts used by PREBORN plus the concepts required for referential closure.",
    "concept_relationship": "Bundled OMOP CONCEPT_RELATIONSHIP rows whose source and target concepts are both included in the PREBORN vocabulary subset.",
    "concept_ancestor": "Bundled OMOP CONCEPT_ANCESTOR hierarchy rows needed to provide ancestry for concepts used by PREBORN.",
    "concept_synonym": "Bundled OMOP CONCEPT_SYNONYM rows for concepts included in the PREBORN vocabulary subset.",
    "vocabulary": "OMOP VOCABULARY metadata for the standardised vocabularies represented in the PREBORN concept bundle.",
    "domain": "OMOP DOMAIN metadata defining the clinical and administrative domains used by PREBORN concepts.",
    "concept_class": "OMOP CONCEPT_CLASS metadata defining concept classifications used by the bundled PREBORN vocabulary.",
    "relationship": "OMOP RELATIONSHIP metadata defining relationship types used by the bundled PREBORN vocabulary.",
    "_msds_concept_map": "Every MSDS mapping-spec resolution (question target, answer value, unit, status) used by this site's build, with its method, model version, cosine and margin (best-effort, unreviewed provenance). Code columns carrying SNOMED directly are self-describing via *_source_concept_id.",
    "_concept_set": "Beyond-MSDS maternity concept set: each concept admitted from the EHR and the anchor concepts and groups that admitted it.",
    "_ehr_provenance": "Per-row mapping provenance for beyond-MSDS EHR facts (method, version, cosine and rule id).",
    "_publish_lock": "Operational lock table used to enforce a single PREBORN publisher or vocabulary rebuilder at a time.",
    "_publish_journal": "Durable PREBORN publish journal used to recover or roll back interrupted multi-table publications.",
}

PREBORN_COLUMN_DESCRIPTIONS = {
    "*": {
        "person_id": "PREBORN-minted identifier for the person represented by the record.",
        "gender_concept_id": "Foreign key to CONCEPT representing the person's gender.",
        "year_of_birth": "Four-digit year of birth derived from the available birth datetime.",
        "month_of_birth": "Month of birth, from 1 to 12, when available.",
        "day_of_birth": "Day of month of birth, from 1 to 31, when available.",
        "birth_datetime": "Recorded or derived date and time of birth.",
        "race_concept_id": "Foreign key to CONCEPT representing the person's race.",
        "ethnicity_concept_id": "Foreign key to CONCEPT representing the person's ethnicity.",
        "location_id": "Optional foreign key to the OMOP LOCATION table.",
        "provider_id": "Optional foreign key to the OMOP PROVIDER table.",
        "care_site_id": "Optional foreign key to the OMOP CARE_SITE table.",
        "person_source_value": "Source-system identifier retained for the person.",
        "gender_source_value": "Verbatim source value describing gender or sex.",
        "gender_source_concept_id": "Concept identifier corresponding to the source gender value, or 0 when unmapped.",
        "race_source_value": "Verbatim source value describing race or NHS ethnic category.",
        "race_source_concept_id": "Concept identifier corresponding to the source race value, or 0 when unmapped.",
        "ethnicity_source_value": "Verbatim source value describing ethnicity.",
        "ethnicity_source_concept_id": "Concept identifier corresponding to the source ethnicity value, or 0 when unmapped.",
        "site_id": "Source-site identifier used to partition and publish PREBORN data.",
        "person_role": "PREBORN role of the person, such as mother or infant.",

        "observation_period_id": "Unique identifier for the person's observation period.",
        "observation_period_start_date": "First date on which the person is represented in PREBORN.",
        "observation_period_end_date": "Last date on which the person is represented in PREBORN.",
        "period_type_concept_id": "Foreign key to CONCEPT describing how the observation period was derived.",

        "condition_occurrence_id": "Unique identifier for a condition occurrence.",
        "condition_concept_id": "Foreign key to the standard CONDITION concept represented by the record.",
        "condition_start_date": "Date on which the condition was recorded or inferred to start.",
        "condition_start_datetime": "Date and time on which the condition was recorded or inferred to start.",
        "condition_end_date": "Date on which the condition ended, when available.",
        "condition_end_datetime": "Date and time on which the condition ended, when available.",
        "condition_type_concept_id": "Foreign key to CONCEPT describing the provenance of the condition record.",
        "condition_status_concept_id": "Foreign key to CONCEPT describing the status of the condition.",
        "stop_reason": "Source or derived reason the condition ended.",
        "visit_occurrence_id": "Optional foreign key to the associated VISIT_OCCURRENCE.",
        "visit_detail_id": "Optional foreign key to the associated VISIT_DETAIL.",
        "condition_source_value": "Verbatim diagnosis or condition value from the source data.",
        "condition_source_concept_id": "Concept identifier corresponding to the source condition value, or 0 when unmapped.",
        "condition_status_source_value": "Verbatim source value describing condition status.",

        "procedure_occurrence_id": "Unique identifier for a procedure occurrence.",
        "procedure_concept_id": "Foreign key to the standard PROCEDURE concept represented by the record.",
        "procedure_date": "Date on which the procedure was performed.",
        "procedure_datetime": "Date and time on which the procedure was performed.",
        "procedure_end_date": "Date on which the procedure ended, when available.",
        "procedure_end_datetime": "Date and time on which the procedure ended, when available.",
        "procedure_type_concept_id": "Foreign key to CONCEPT describing the provenance of the procedure record.",
        "modifier_concept_id": "Foreign key to CONCEPT for an optional procedure modifier.",
        "quantity": "Number of procedures or units represented by the record.",
        "procedure_source_value": "Verbatim procedure value from the source data.",
        "procedure_source_concept_id": "Concept identifier corresponding to the source procedure value, or 0 when unmapped.",
        "modifier_source_value": "Verbatim source value for the procedure modifier.",

        "measurement_id": "Unique identifier for a measurement.",
        "measurement_concept_id": "Foreign key to the standard MEASUREMENT concept represented by the record.",
        "measurement_date": "Date on which the measurement was taken.",
        "measurement_datetime": "Date and time on which the measurement was taken.",
        "measurement_time": "Source measurement time when supplied separately from the date.",
        "measurement_type_concept_id": "Foreign key to CONCEPT describing the provenance of the measurement.",
        "operator_concept_id": "Foreign key to CONCEPT representing a comparison operator such as less than or greater than.",
        "value_as_number": "Numeric value of the measurement or observation.",
        "value_as_concept_id": "Foreign key to CONCEPT representing a coded result value.",
        "unit_concept_id": "Foreign key to CONCEPT representing the standard unit.",
        "range_low": "Lower bound of the source reference range.",
        "range_high": "Upper bound of the source reference range.",
        "measurement_source_value": "Verbatim measurement code or label from the source data.",
        "measurement_source_concept_id": "Concept identifier corresponding to the source measurement value, or 0 when unmapped.",
        "unit_source_value": "Verbatim unit supplied by the source data.",
        "unit_source_concept_id": "Concept identifier corresponding to the source unit, or 0 when unmapped.",
        "value_source_value": "Verbatim source representation of the result value.",
        "measurement_event_id": "Identifier of the event to which this measurement is linked.",
        "meas_event_field_concept_id": "Concept identifying the event table field referenced by measurement_event_id.",

        "observation_id": "Unique identifier for an observation.",
        "observation_concept_id": "Foreign key to the standard OBSERVATION concept represented by the record.",
        "observation_date": "Date on which the observation was recorded.",
        "observation_datetime": "Date and time on which the observation was recorded.",
        "observation_type_concept_id": "Foreign key to CONCEPT describing the provenance of the observation.",
        "value_as_string": "Text value of the observation when it is not represented numerically or as a concept.",
        "qualifier_concept_id": "Foreign key to CONCEPT qualifying the observation value.",
        "observation_source_value": "Verbatim observation or finding value from the source data.",
        "observation_source_concept_id": "Concept identifier corresponding to the source observation value, or 0 when unmapped.",
        "qualifier_source_value": "Verbatim source value for the observation qualifier.",
        "observation_event_id": "Identifier of the event to which this observation is linked.",
        "obs_event_field_concept_id": "Concept identifying the event table field referenced by observation_event_id.",

        "visit_concept_id": "Foreign key to CONCEPT representing the standard visit type.",
        "visit_start_date": "Date on which the visit started.",
        "visit_start_datetime": "Date and time on which the visit started.",
        "visit_end_date": "Date on which the visit ended.",
        "visit_end_datetime": "Date and time on which the visit ended.",
        "visit_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit.",
        "visit_source_value": "Verbatim care-contact or visit value from the source data.",
        "visit_source_concept_id": "Concept identifier corresponding to the source visit value, or 0 when unmapped.",
        "admitted_from_concept_id": "Foreign key to CONCEPT representing where the person was admitted from.",
        "admitted_from_source_value": "Verbatim source value describing where the person was admitted from.",
        "discharged_to_concept_id": "Foreign key to CONCEPT representing the discharge destination.",
        "discharged_to_source_value": "Verbatim source value describing the discharge destination.",
        "preceding_visit_occurrence_id": "Identifier of the immediately preceding visit occurrence for the person.",

        "visit_detail_id": "Unique identifier for a detailed visit segment.",
        "visit_detail_concept_id": "Foreign key to CONCEPT representing the standard visit-detail type.",
        "visit_detail_start_date": "Date on which the visit detail started.",
        "visit_detail_start_datetime": "Date and time on which the visit detail started.",
        "visit_detail_end_date": "Date on which the visit detail ended.",
        "visit_detail_end_datetime": "Date and time on which the visit detail ended.",
        "visit_detail_type_concept_id": "Foreign key to CONCEPT describing the provenance of the visit detail.",
        "visit_detail_source_value": "Verbatim source value describing the visit detail.",
        "visit_detail_source_concept_id": "Concept identifier corresponding to the source visit-detail value, or 0 when unmapped.",
        "preceding_visit_detail_id": "Identifier of the immediately preceding visit detail for the person.",
        "parent_visit_detail_id": "Identifier of the containing visit detail, when nested.",

        "episode_id": "Unique identifier for a clinical episode.",
        "episode_concept_id": "Foreign key to CONCEPT representing the standard episode type.",
        "episode_start_date": "Date on which the episode started.",
        "episode_start_datetime": "Date and time on which the episode started.",
        "episode_end_date": "Date on which the episode ended.",
        "episode_end_datetime": "Date and time on which the episode ended.",
        "episode_parent_id": "Identifier of a parent episode, when episodes are nested.",
        "episode_number": "Sequence number of the episode for the person.",
        "episode_object_concept_id": "Foreign key to CONCEPT identifying the domain represented by the episode.",
        "episode_type_concept_id": "Foreign key to CONCEPT describing how the episode was derived.",
        "episode_source_value": "Verbatim source value used to derive the episode.",
        "episode_source_concept_id": "Concept identifier corresponding to the source episode value, or 0 when unmapped.",
        "event_id": "Identifier of the clinical event linked to the episode.",
        "episode_event_field_concept_id": "Concept identifying the event-table field referenced by event_id.",

        "domain_concept_id_1": "Concept identifying the domain of the first related fact.",
        "fact_id_1": "Identifier of the first fact in the relationship.",
        "domain_concept_id_2": "Concept identifying the domain of the second related fact.",
        "fact_id_2": "Identifier of the second fact in the relationship.",
        "relationship_concept_id": "Foreign key to CONCEPT describing the relationship between the two facts.",

        "pregnancy_id": "PREBORN-minted identifier for the pregnancy linked to the record.",
        "pregnancy_start_date": "Estimated start date of the pregnancy.",
        "pregnancy_end_date": "Date on which the pregnancy ended.",
        "gestational_length_in_day": "Gestational length in completed days.",
        "pregnancy_outcome_concept_id": "Foreign key to CONCEPT representing the pregnancy outcome.",
        "pregnancy_mode_delivery_concept_id": "Foreign key to CONCEPT representing the mode of delivery.",
        "pregnancy_single_concept_id": "Foreign key to CONCEPT indicating singleton or multiple pregnancy status.",
        "pregnancy_marital_status_concept_id": "Foreign key to CONCEPT representing marital status during pregnancy.",
        "pregnancy_number_fetuses": "Number of fetuses recorded for the pregnancy.",
        "pregnancy_number_liveborn": "Number of liveborn infants recorded for the pregnancy.",
        "prev_pregnancy_parity_concept_id": "Concept representing previous pregnancy parity.",
        "prev_pregnancy_gravidity": "Number of previous pregnancies.",
        "prev_livebirth_number": "Number of previous live births.",
        "prev_still_misc_number": "Number of previous stillbirths or miscarriages.",
        "pre_pregnancy_bmi": "Body mass index recorded or derived before pregnancy.",
        "pregnancy_folic_concept_id": "Concept indicating folic-acid supplementation.",
        "pregnancy_outcome_source_value": "Verbatim source value for pregnancy outcome.",
        "pregnancy_mode_delivery_source_value": "Verbatim source value for mode of delivery.",
        "pre_pregnancy_bmi_source": "Source system or field used for the pre-pregnancy BMI.",
        "pregnancy_marital_status_source_value": "Verbatim source value for marital status during pregnancy.",

        "infant_id": "PREBORN-minted identifier for the infant and corresponding infant person.",
        "birth_outcome_concept_id": "Foreign key to CONCEPT representing the infant birth outcome.",
        "birth_weight": "Recorded birth weight, in grams where supplied by the source.",
        "birth_con_malformation_concept_id": "Concept indicating whether a congenital malformation was recorded.",
        "birth_apgar": "Recorded five-minute Apgar score.",
        "labour_delivery_id": "Source labour or delivery identifier linking the infant to delivery records.",
        "birth_weight_source": "Source system or field used for birth weight.",
        "birth_apgar_source": "Source system or field used for the Apgar score.",

        "source_table": "Name of the source table or transformation arm that produced the record.",
        "enrichment_tag": "Marker distinguishing enrichment-derived rows from common-core rows.",

        "cdm_source_name": "Name of the PREBORN CDM instance.",
        "cdm_source_abbreviation": "Short abbreviation for the PREBORN CDM instance.",
        "cdm_holder": "Organisation responsible for the PREBORN CDM instance.",
        "source_description": "Description of the source data represented by the CDM.",
        "source_documentation_reference": "Reference to documentation describing the source data.",
        "cdm_etl_reference": "Reference or version identifying the PREBORN ETL implementation.",
        "source_release_date": "Release date of the source data used by the ETL.",
        "cdm_release_date": "Release date of the generated PREBORN CDM.",
        "cdm_version": "OMOP Common Data Model version implemented by PREBORN.",
        "cdm_version_concept_id": "Concept identifier representing the OMOP CDM version.",
        "vocabulary_version": "Version of the OMOP vocabulary bundle loaded into PREBORN.",

        "concept_id": "Unique OMOP identifier for the concept.",
        "concept_name": "Unambiguous, human-readable name of the concept.",
        "domain_id": "Identifier of the OMOP domain to which the concept belongs.",
        "vocabulary_id": "Identifier of the source vocabulary from which the concept was drawn.",
        "concept_class_id": "Identifier of the class used to categorise the concept.",
        "standard_concept": "Flag indicating a Standard concept, Classification concept or source concept.",
        "concept_code": "Code assigned to the concept by its source vocabulary.",
        "valid_start_date": "Date on which the vocabulary record became valid.",
        "valid_end_date": "Date on which the vocabulary record ceased to be valid.",
        "invalid_reason": "Reason the vocabulary record was invalidated; null for a currently valid record.",
        "concept_id_1": "Identifier of the source concept in a directional concept relationship.",
        "concept_id_2": "Identifier of the target concept in a directional concept relationship.",
        "relationship_id": "Identifier of the relationship type.",
        "ancestor_concept_id": "Identifier of the ancestor concept in the OMOP hierarchy.",
        "descendant_concept_id": "Identifier of the descendant concept in the OMOP hierarchy.",
        "min_levels_of_separation": "Minimum number of hierarchy levels between ancestor and descendant.",
        "max_levels_of_separation": "Maximum number of hierarchy levels between ancestor and descendant.",
        "concept_synonym_name": "Alternative name or synonym for the concept.",
        "language_concept_id": "Concept identifier representing the synonym language.",
        "vocabulary_name": "Human-readable name of the vocabulary.",
        "vocabulary_reference": "Reference describing the vocabulary or its source.",
        "vocabulary_version": "Loaded version of the vocabulary.",
        "vocabulary_concept_id": "Concept identifier representing the vocabulary.",
        "domain_name": "Human-readable name of the OMOP domain.",
        "domain_concept_id": "Concept identifier representing the domain.",
        "concept_class_name": "Human-readable name of the concept class.",
        "concept_class_concept_id": "Concept identifier representing the concept class.",
        "relationship_name": "Human-readable name of the relationship type.",
        "is_hierarchical": "Flag indicating whether the relationship is hierarchical.",
        "defines_ancestry": "Flag indicating whether the relationship contributes to CONCEPT_ANCESTOR.",
        "reverse_relationship_id": "Identifier of the relationship used in the reverse direction.",

        "death_date": "Date of death; the earliest recorded when MSDS holds more than one.",
        "death_datetime": "Date and time of death; the earliest recorded when MSDS holds more than one.",
        "death_type_concept_id": "Foreign key to CONCEPT describing the provenance of the death record (EHR 32817).",
        "cause_concept_id": "Foreign key to CONCEPT for the cause of death; 0 because MSDS carries no cause.",
        "cause_source_value": "Verbatim cause of death; NULL because MSDS carries no cause.",
        "cause_source_concept_id": "Concept identifier for the source cause of death; 0 when unmapped.",

        "address_1": "Street address line 1; always NULL (PREBORN holds LSOA only).",
        "address_2": "Street address line 2; always NULL (PREBORN holds LSOA only).",
        "city": "City; always NULL (PREBORN holds LSOA only).",
        "state": "State; always NULL (not applicable in the UK).",
        "zip": "Postal code; always NULL (the postcode is never published).",
        "county": "ONS local authority district code (ONSPD oslaua) containing the LSOA.",
        "location_source_value": "ONS LSOA 2011 code of the residence (ONSPD lsoa11, the vintage of the silver IMD deciles).",
        "country_concept_id": "Foreign key to CONCEPT for the country (United Kingdom 42035286).",
        "country_source_value": "Country code of the location ('GB').",
        "latitude": "Latitude; always NULL (a postcode centroid would re-identify the address).",
        "longitude": "Longitude; always NULL (a postcode centroid would re-identify the address).",

        "rule_id": "Mapping-spec or pipeline rule that produced the row.",
        "src_code": "Source code value as recorded in MSDS.",
        "src_text": "Source code description or answer text that was mapped.",
        "method": "How the concept was chosen (spec, snomed_maps_to, embed, ...).",
        "model_version": "Embedding model version for embed-derived mappings; NULL otherwise.",
        "cosine": "Cosine similarity of the chosen concept for embed-derived mappings.",
        "margin": "Cosine margin over the runner-up concept for embed-derived mappings.",
        "rule_ref": "Reference to the spec rule or lookup row the mapping came from.",
        "role": "Role of the concept: target/value/unit/status in _msds_concept_map; anchor/descendant in _concept_set.",
        "anchors": "Anchor concept ids that admitted the concept.",
        "groups": "Maternity concept groups the concept belongs to.",
        "cdm_table": "CDM table holding the row.",
        "cdm_id": "Primary key of the row in cdm_table.",
        "map_method": "Mapping method used for the row.",
        "map_version": "Mapping model or rule-set version.",
        "map_cosine": "Mapping cosine similarity, when embed-derived.",
        "map_rule_id": "Mapping rule id, when rule-derived.",
        "detail": "Human-readable diagnostic, publication or recovery detail.",

        "lock_name": "Name of the operational lock.",
        "token": "Unique token owned by the process holding the lock.",
        "purpose": "Description of the operation protected by the lock.",
        "acquired_at": "Timestamp at which the lock was acquired.",
        "publish_token": "Unique token identifying one publication attempt.",
        "table_name": "Name of the table included in the publication attempt.",
        "version_before": "Delta table version recorded before publication for rollback.",
        "state": "Current state of the publication journal entry.",
        "started_at": "Timestamp at which publication of the table began.",
        "finished_at": "Timestamp at which publication or recovery finished.",
    },
    "person": {
        "person_id": "Primary key for a PREBORN person; minted deterministically within the source site."
    },
    "location": {
        "location_id": "Primary key for an LSOA location: 8e15 + country rank (E01=1, W01=2) x 1e9 + the LSOA digits (the OGR scheme); one row per LSOA per site."
    },
    "pregnancy": {
        "person_id": "Identifier of the pregnant person linked to this pregnancy.",
        "pregnancy_id": "Primary key for a PREBORN pregnancy; minted deterministically within the source site."
    },
    "infant": {
        "pregnancy_id": "Identifier of the pregnancy that produced the infant.",
        "infant_id": "Primary key for the infant extension and identifier of the corresponding infant PERSON row."
    },
    "episode_event": {
        "episode_id": "Identifier of the pregnancy episode containing the linked event."
    },
    "_publish_journal": {
        "site_id": "Source site being published by this journal entry."
    },
}


def apply_table_comments(schema, tables):
    """Write the descriptions above to schema's tables and columns, skipping any that are already current."""
    catalog, schema_name = schema.split(".", 1)
    table_list = ", ".join(f"'{sql_str(t)}'" for t in tables)
    current_table_comments = {
        row["table_name"]: row["comment"] for row in spark.sql(f"""
          SELECT table_name, comment FROM `{catalog}`.information_schema.tables
          WHERE table_schema = '{sql_str(schema_name)}' AND table_name IN ({table_list})""").collect()}
    column_rows = spark.sql(f"""
      SELECT table_name, column_name, comment FROM `{catalog}`.information_schema.columns
      WHERE table_schema = '{sql_str(schema_name)}' AND table_name IN ({table_list})
      ORDER BY table_name, ordinal_position""").collect()

    updates = 0
    for table in tables:
        if table not in current_table_comments:
            continue                                  # the table does not exist in this schema
        wanted = PREBORN_TABLE_COMMENTS.get(table, f"PREBORN table containing {table.replace('_', ' ')} records.")
        if current_table_comments[table] != wanted:
            spark.sql(f"COMMENT ON TABLE `{catalog}`.`{schema_name}`.`{table}` IS '{sql_str(wanted)}'")
            updates += 1
        for row in column_rows:
            if row["table_name"] != table:
                continue
            column = row["column_name"]
            wanted = (PREBORN_COLUMN_DESCRIPTIONS.get(table, {}).get(column)
                      or PREBORN_COLUMN_DESCRIPTIONS["*"].get(column)
                      or f"PREBORN {table.replace('_', ' ')} attribute: {column.replace('_', ' ')}.")
            if row["comment"] != wanted:
                spark.sql(f"ALTER TABLE `{catalog}`.`{schema_name}`.`{table}` ALTER COLUMN `{column}` "
                          f"COMMENT '{sql_str(wanted)}'")
                updates += 1
    print("descriptions updated:", updates)

### The writer lock and the publish journal

In [0]:
def create_publish_controls(target):
    """Create target._publish_lock and target._publish_journal if missing."""
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_lock (
      lock_name STRING, token STRING, purpose STRING, acquired_at TIMESTAMP) USING DELTA""")
    spark.sql(f"""CREATE TABLE IF NOT EXISTS {target}._publish_journal (
      publish_token STRING, site_id STRING, table_name STRING, version_before BIGINT,
      state STRING, started_at TIMESTAMP, finished_at TIMESTAMP, detail STRING) USING DELTA""")
    apply_table_comments(target, ["_publish_lock", "_publish_journal"])


def acquire_write_lock(target, purpose):
    """Take the target's writer lock and return its token; fail if another writer holds it."""
    create_publish_controls(target)
    token = uuid.uuid4().hex
    # The publisher refreshes the lock after every table, so a lock untouched for two hours is abandoned.
    spark.sql(f"""DELETE FROM {target}._publish_lock
      WHERE lock_name = 'exclusive' AND acquired_at < current_timestamp() - INTERVAL 2 HOURS""")
    spark.sql(f"""INSERT INTO {target}._publish_lock
      VALUES ('exclusive', '{token}', '{sql_str(purpose)}', current_timestamp())""")
    holders = spark.sql(f"SELECT token FROM {target}._publish_lock WHERE lock_name = 'exclusive'").collect()
    if len(holders) != 1 or holders[0]["token"] != token:
        spark.sql(f"DELETE FROM {target}._publish_lock WHERE token = '{token}'")
        raise RuntimeError(f"another PREBORN writer holds the lock on {target}")
    return token


def refresh_write_lock(target, token):
    """Show the lock is still in use (see acquire_write_lock)."""
    spark.sql(f"""UPDATE {target}._publish_lock SET acquired_at = current_timestamp()
      WHERE lock_name = 'exclusive' AND token = '{token}'""")


def release_write_lock(target, token):
    """Give up the writer lock taken by acquire_write_lock."""
    spark.sql(f"DELETE FROM {target}._publish_lock WHERE token = '{token}'")


def restore_versions(target, versions):
    """Restore each target table to its recorded version, newest first. Returns any error messages."""
    errors = []
    for table, version in reversed(list(versions.items())):
        try:
            spark.sql(f"RESTORE TABLE {target}.{table} TO VERSION AS OF {version}")
        except Exception as error:
            errors.append(f"{table}: {type(error).__name__}: {error}")
    return errors


def set_journal_state(target, token, state, detail):
    """Mark every journal row of one publish with its final state."""
    spark.sql(f"""UPDATE {target}._publish_journal
      SET state = '{sql_str(state)}', finished_at = current_timestamp(), detail = '{sql_str(detail)[:8000]}'
      WHERE publish_token = '{sql_str(token)}'""")


def recover_interrupted_publishes(target):
    """Restore any earlier publish that stopped before it was marked COMMITTED."""
    rows = spark.sql(f"""SELECT publish_token, table_name, version_before FROM {target}._publish_journal
      WHERE state IN ('IN_PROGRESS', 'RECOVERY_FAILED')
      ORDER BY started_at, publish_token, table_name""").collect()
    tokens = []
    for row in rows:
        if row["publish_token"] not in tokens:
            tokens.append(row["publish_token"])
    for old_token in tokens:
        versions = {row["table_name"]: int(row["version_before"]) for row in rows if row["publish_token"] == old_token}
        errors = restore_versions(target, versions)
        if errors:
            set_journal_state(target, old_token, "RECOVERY_FAILED", "; ".join(errors))
            raise RuntimeError(f"failed to recover interrupted PREBORN publish {old_token}: {errors}")
        set_journal_state(target, old_token, "RECOVERED", "restored automatically before next writer")
        print("recovered interrupted PREBORN publish", old_token)

In [0]:
def publish(target, tables, check_after_publish):
    """Copy this site's rows of `tables` from PUB to target, rebuild the target's vocabulary, run
    check_after_publish(), and commit; on any failure restore every touched table and re-raise."""
    create_cdm_tables(target)
    apply_table_comments(target, list(PREBORN_TABLE_COMMENTS))
    token = acquire_write_lock(target, f"publish {SITE_ID}")
    versions = {}
    journaled = False
    try:
        recover_interrupted_publishes(target)

        # Record where every table we may touch stands now.
        versions = {t: latest_version(f"{target}.{t}") for t in list(tables) + ["cdm_source"] + VOCAB_TABLES}
        journal_rows = ",\n".join(
            f"('{token}', '{SITE}', '{t}', {v}, 'IN_PROGRESS', current_timestamp(), NULL, NULL)"
            for t, v in versions.items())
        spark.sql(f"""INSERT INTO {target}._publish_journal
          (publish_token, site_id, table_name, version_before, state, started_at, finished_at, detail)
          VALUES {journal_rows}""")
        journaled = True

        # Replace this site's rows, table by table, and check the counts match.
        for table in tables:
            columns = ", ".join(table_columns(target, table))
            spark.sql(f"""INSERT INTO {target}.{table} BY NAME
              REPLACE WHERE site_id = '{SITE}'
              SELECT {columns} FROM {PUB}.{table} WHERE site_id = '{SITE}'""")
            expected = scalar(f"SELECT count(*) FROM {PUB}.{table} WHERE site_id = '{SITE}'")
            actual = scalar(f"SELECT count(*) FROM {target}.{table} WHERE site_id = '{SITE}'")
            if actual != expected:
                raise RuntimeError(f"publish count mismatch {table}: target {actual}, staged {expected}")
            refresh_write_lock(target, token)
            print(f"  published {table}: {actual} rows")

        spark.sql(f"INSERT INTO {target}.cdm_source BY NAME REPLACE WHERE true SELECT * FROM {PUB}.cdm_source")
        refresh_write_lock(target, token)

        build_bundled_vocabulary(target)
        apply_table_comments(target, VOCAB_TABLES)
        refresh_write_lock(target, token)

        check_after_publish()
        set_journal_state(target, token, "COMMITTED", "publish and post-validation succeeded")
        print("published", PUB, "->", target, "for site", SITE_ID)
    except Exception as error:
        rollback_errors = restore_versions(target, versions) if versions else []
        if journaled:
            if rollback_errors:
                set_journal_state(target, token, "RECOVERY_FAILED",
                                  f"publish failed: {type(error).__name__}: {error}; rollback errors: {rollback_errors}")
            else:
                set_journal_state(target, token, "ROLLED_BACK",
                                  f"publish failed and was restored: {type(error).__name__}: {error}")
        if rollback_errors:
            raise RuntimeError(f"publish failed ({type(error).__name__}: {error}); "
                               f"rollback errors: {rollback_errors}") from error
        raise
    finally:
        try:
            release_write_lock(target, token)
        except Exception as lock_error:
            print("WARNING: could not release the publish lock; it will be treated as abandoned after two hours:",
                  lock_error)


publish(TARGET, BEYOND_PUBLISH_TABLES, check_after_publish=lambda: run_qa(TARGET, "published"))

### Tidy up

The working schemas are dropped, unless `keep_working_schemas` is `yes`.

In [0]:
if KEEP_WORKING_SCHEMAS:
    print("kept working schemas", PUB, "and", STG)
else:
    spark.sql(f"DROP SCHEMA IF EXISTS {STG} CASCADE")
    spark.sql(f"DROP SCHEMA IF EXISTS {PUB} CASCADE")
    print("dropped working schemas for run", RUN_TAG)
print("PREBORN Barts extension published to", TARGET)